# Imports

In [ ]:
#%matplotlib ipympl
import matplotlib.pyplot as plt
import numpy as np
import glob, re
#from slab import fitexp, SlabFile, get_next_filename
from datamanagement import *

# Cable Attenuation

In [ ]:
file_path = '00000_20260326_RfSoc_Cal_Cable0.01GHzCF_2.0MHzSpan.h5'
with SlabFile('data/20260327/' + file_path, 'r') as f:
        #print(f.keys())
        #phase=np.array(f['phases'][:])[0]
        freq=np.array(f['fpts'][:])[0]
        mag_S21=np.array(f['mags'][:])[0]
        #rodp = np.array(f['Rod_Position'])[0]
        #avgs = np.array(f['Averages'])

ind = np.where(freq/1e6 >= 13.5)[0][0]
cable_atten_dB = np.round(mag_S21[ind], 4)
print(f'freq axis crosses 13.5 MHz at ind {ind}, corresponding to freq {freq[ind]/1e6} MHZ')
print(f'The attenuation at (ind: {ind}, freq: {freq[ind]/1e6} MHz) is: {cable_atten_dB} dB')

plt.figure(dpi = 300)
plt.plot(freq/1e6, mag_S21)
plt.xlabel('Freq (MHz)')
plt.ylabel('dB')
plt.title('Cable Attenuation - 13.5MHZ')
plt.show()

# ADC

## Functions

In [ ]:
def lsb_to_dBm(adc_code, N=12, Vref=2.0, R=50):
    """
    Convert ADC code to power in dBm.
    
    Parameters:
        adc_code : int or np.array
            ADC digital value (can be signed or unsigned)
        N        : int
            ADC resolution in bits
        Vref     : float
            Peak-to-peak voltage of ADC reference (V)
        R        : float
            Load resistance (Ohms)
    
    Returns:
        float or np.array : Power in dBm
    """
    V_LSB = Vref / (2**N)
    V_rms = (adc_code * V_LSB) / np.sqrt(2)  # convert to RMS assuming sinusoid
    P_watt = V_rms**2 / R
    P_dBm = 10 * np.log10(P_watt / 1e-3)
    return P_dBm

def signed_adc_to_dBm(adc_code, N=12, Vref=2.0, R=50):
    """
    Convert a signed ADC code (centered at 0) to power in dBm.

    Parameters:
        adc_code : int or np.array
            ADC digital value (signed, from -2^(N-1) to 2^(N-1)-1)
        N        : int
            ADC resolution in bits
        Vref     : float
            Full-scale voltage (peak-to-peak) of ADC reference (V)
        R        : float
            Load resistance in Ohms (default 50 Ω)

    Returns:
        float or np.array : Power in dBm
    """
    # Convert ADC code to voltage (centered at 0)
    V_LSB = Vref / (2**N)
    V = adc_code * V_LSB

    # Convert to RMS voltage (assuming sinusoidal signal)
    V_rms = V / np.sqrt(2)

    # Compute power in watts
    P_watt = V_rms**2 / R

    # Convert to dBm
    P_dBm = 10 * np.log10(P_watt / 1e-3)
    return P_dBm

In [ ]:
# Example usage:
adc_value = 40/(2**4)        # signed ADC code
power_dBm = signed_adc_to_dBm(adc_value, N=11, Vref=0.5, R=50)
print(f"ADC value {adc_value} corresponds to {power_dBm:.2f} dBm")

## Calibration Data

In [ ]:
# -----------------------------
# System Parameters
# -----------------------------
Fs_orig = 4096e6
D = 2
Fs = Fs_orig/D
#N = 65536
N_bits = 16
FS = 2**(N_bits-1)  ## -1 for signed number
f_tone = 13.5e6

In [ ]:
# List of file paths and labels
date = '20260327'
file_paths = [
    '00000_' + date + '_ADC+PFB_Cal-40dBm.h5',
    '00000_' + date + '_ADC+PFB_Cal-38dBm.h5',
    '00000_' + date + '_ADC+PFB_Cal-35dBm.h5',
    '00000_' + date + '_ADC+PFB_Cal-32dBm.h5',
    '00000_' + date + '_ADC+PFB_Cal-30dBm.h5',
    '00000_' + date + '_ADC+PFB_Cal-28dBm.h5',
    '00000_' + date + '_ADC+PFB_Cal-25dBm.h5',
    '00000_' + date + '_ADC+PFB_Cal-22dBm.h5',
    '00000_' + date + '_ADC+PFB_Cal-20dBm.h5',
    '00000_' + date + '_ADC+PFB_Cal-18dBm.h5',
    '00000_' + date + '_ADC+PFB_Cal-15dBm.h5',
    '00000_' + date + '_ADC+PFB_Cal-12dBm.h5',
    '00000_' + date + '_ADC+PFB_Cal-10dBm.h5',
    '00000_' + date + '_ADC+PFB_Cal-8dBm.h5',
    '00000_' + date + '_ADC+PFB_Cal-5dBm.h5',
    '00000_' + date + '_ADC+PFB_Cal0dBm.h5',
    '00000_' + date + '_ADC+PFB_Cal5dBm.h5',
    '00000_' + date + '_ADC+PFB_Cal6dBm.h5',
    '00000_' + date + '_ADC+PFB_Cal7dBm.h5',
    '00000_' + date + '_ADC+PFB_Cal8dBm.h5',
    '00000_' + date + '_ADC+PFB_Cal10dBm.h5',
]#
labels_list  = ['-40', '-38', '-35', '-32', '-30', '-28', '-25', '-22', '-20', '-18', '-15', '-12', '-10', '-8', '-5', '0', '+5', '+6', '+7', '+8', '+10']

# Prepare colors
colors = plt.cm.viridis(np.linspace(0, 1, len(file_paths)))

plt.figure(figsize=(12, 8))

# --- Process each measurement ---
for i, file_path in enumerate(file_paths):
    # Load complex data

    with SlabFile('data/' + date + '/' + file_path, 'r') as f:
        xi_adc = np.array(f['xi_ADC'])[0]
        xq_adc = np.array(f['xq_ADC'])[0]
        xi_pfb = np.array(f['xi_PFB'])[0]
        xq_pfb = np.array(f['xq_PFB'])[0]
        fs_adc = np.array(f['ADC fs'])[0]
        fs_pfb = np.array(f['PFB fs'])[0]
        qout = np.array(f['Qout'])[0]
    #print(f'Comparing Fs {Fs/1e6} MHz with fs_adc {fs_adc} MHz from board')
    # n = np.arange(len(xi_adc))
    # t = n/Fs
    # plt.figure( dpi = 150)
    # plt.plot(t,xi_adc,  label = 'xi')
    # plt.plot(t, xq_adc, label = 'xq')
    # plt.legend()
    # plt.ylabel('ADC Buffer Xi and Xq')
    # plt.xlabel('time [us]')
    # plt.show()
    x_adc_t = xi_adc + 1j*xq_adc
    x_adc_t = np.array(x_adc_t, dtype=np.complex128)

    N = len(x_adc_t)

    # --- 1. Window Processing ---
    window = np.hanning(N)
    x_windowed = x_adc_t * window

    # --- 2. Complex FFT ---
    fft_complex = np.fft.fft(x_windowed) / N
    fft_complex = np.fft.fftshift(fft_complex)
    freqs = np.fft.fftshift(np.fft.fftfreq(N, 1/Fs))

    # --- 3. Amplitude in dBFS ---
    mag_linear = np.abs(fft_complex) / 0.5  # Correct for Hanning CG
    with np.errstate(divide='ignore'):
        mag_dbfs = 20 * np.log10(mag_linear / FS)

    # --- 4. PSD ---
    enbw_hz = 1.5 * (Fs / N)
    psd_dbfs_hz = mag_dbfs - 10 * np.log10(enbw_hz)

    # --- 5. Plot amplitude ---
    plt.subplot(2, 1, 1)
    plt.plot(freqs / 1e6, mag_dbfs, color=colors[i], label=labels_list[i], zorder = len(labels_list) - i)

    # --- 6. Plot PSD ---
    plt.subplot(2, 1, 2)
    plt.plot(freqs / 1e6, psd_dbfs_hz, color=colors[i], label=labels_list[i], zorder = len(labels_list) - i)

# --- Finalize Amplitude plot ---
plt.subplot(2, 1, 1)
plt.title(f"Complex Spectrum (I/Q) | Fs = {Fs/1e6:.2f} MHz")
plt.ylabel("Amplitude [dBFS]")
plt.grid(True, alpha=0.4)
plt.ylim([None, 5])
#plt.legend()

# --- Finalize PSD plot ---
plt.subplot(2, 1, 2)
plt.xlabel("Frequency [MHz]")
plt.ylabel("PSD [dBFS/Hz]")
plt.grid(True, alpha=0.4)
#plt.legend()

plt.tight_layout()
plt.savefig('images/' + date + '/ADC_zcu111_AmpdBFS_and_PSD.png',dpi=300)
plt.show()

# --- Print peak info for each measurement ---
dBFS_measured = []
for i, file_path in enumerate(file_paths):
    with SlabFile('data/' + date + '/' + file_path, 'r') as f:
        xi_adc = np.array(f['xi_ADC'])[0]
        xq_adc = np.array(f['xq_ADC'])[0]
        xi_pfb = np.array(f['xi_PFB'])[0]
        xq_pfb = np.array(f['xq_PFB'])[0]
        fs_adc = np.array(f['ADC fs'])[0]
        fs_pfb = np.array(f['PFB fs'])[0]
        qout = np.array(f['Qout'])[0]
    #print(f'Comparing Fs {Fs/1e6} MHz with fs_adc {fs_adc} MHz from board')

    x_adc_t = xi_adc + 1j*xq_adc
    x_adc_t = np.array(x_adc_t, dtype=np.complex128)
    N = len(x_adc_t)
    window = np.hanning(N)
    x_windowed = x_adc_t * window
    fft_complex = np.fft.fft(x_windowed) / N
    mag_linear = np.abs(fft_complex) / 0.5
    mag_dbfs = 20 * np.log10(mag_linear / FS)
    peak_idx = np.argmax(mag_dbfs)
    dBFS_measured.append(np.round(mag_dbfs[peak_idx], 2))
    print(f"{labels_list[i]}: Peak Freq = {freqs[peak_idx]}, Peak = {mag_dbfs[peak_idx]:.2f} dBFS, Peak linear = {mag_linear[peak_idx]:.3f}")

In [ ]:
# Measured data
P_dBm = np.array([-40, -38, -35, -32, -30, -28, -25, -22, -20, -18, -15, -12, -10, -8, -5, 0, +5, +6, +7, +8, +10])          # Generator power in dBm. 
dBFS_measured = np.array(dBFS_measured)  # Peak FFT measured in dBFS

# Linear fit (dBFS = P_dBm + K)
K = dBFS_measured[1] - P_dBm[1]  # Calibration constant
dBFS_fit = P_dBm + K

# Graficar
plt.figure(figsize=(7,5))
plt.plot(P_dBm, dBFS_measured, 'o', label='Measurements')
plt.plot(P_dBm, dBFS_fit, '--', label=f'Linear fit: dBFS = dBm + {K:.2f}')
plt.xlabel('Generator power (dBm)')
plt.ylabel('ADC Level (dBFS)')
plt.title('Calibration dBm vs dBFS')
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
# Measured data
P_dBm = np.array([-40, -38, -35, -32, -30, -28, -25, -22, -20, -18, -15, -12, -10, -8, -5, 0, +5, +6, +7, +8, +10])          # Generator power in dBm. 
dBFS_measured = np.array(dBFS_measured)  # Pico FFT medido en dBFS

# Calibration constant (linear relation: dBFS = P_dBm + K)
K = dBFS_measured[1] - P_dBm[1]  # ~-12.83 dB
dBFS_fit = P_dBm + K

# ADC full-scale RMS voltage
V_FS_RMS = 0.354  # V RMS for differential input (=0.5V/sqrt(2))

# Convert dBFS to RMS voltage
V_RMS_measured = V_FS_RMS * 10**(dBFS_measured/20)
V_RMS_fit = V_FS_RMS * 10**(dBFS_fit/20)

# Plot 1: Generator power vs ADC level in dBFS
plt.figure(figsize=(7,5))
plt.plot(P_dBm, dBFS_measured, 'o', label='Measured points')
plt.plot(P_dBm, dBFS_fit, '--', label=f'Linear fit: dBFS = dBm + {K:.2f}')
plt.xlabel('Input Power (dBm)')
plt.ylabel('ADC Level (dBFS)')
plt.title('Calibration: dBm → dBFS')
plt.grid(True)
plt.legend()
plt.savefig('images/' + date + '/Lab_B_ADC_zcu111_dBFS_to_dBm.png',dpi=300)

# Plot 2: Generator power vs ADC RMS voltage
plt.figure(figsize=(7,5))
plt.plot(P_dBm, V_RMS_measured*1e3, 'o', label='Measured RMS voltage')
plt.plot(P_dBm, V_RMS_fit*1e3, '--', label='Linear fit')
plt.xlabel('Input Power (dBm)')
plt.ylabel('RMS Voltage at ADC (mV)')
plt.title('Calibration: dBm → RMS Voltage')
plt.grid(True)
plt.legend()
plt.xlim(-42,15)
plt.savefig('images/' + date + '/Lab_B_ADC_zcu111_RMS_to_dBm.png',dpi=300)
plt.show()
print(f"Calibration constant K (dBFS = dBm + K): {K:.2f} dB")

## One File Analysis

In [ ]:
Pin='-10'
date = '20260327'
with SlabFile('data/' + date + '/' + '00000_' + date + '_ADC+PFB_Cal' + Pin + 'dBm.h5', 'r') as f:
        xi_adc = np.array(f['xi_ADC'])[0]
        xq_adc = np.array(f['xq_ADC'])[0]
        xi_pfb = np.array(f['xi_PFB'])[0]
        xq_pfb = np.array(f['xq_PFB'])[0]
        fs_adc = np.array(f['ADC fs'])[0]
        fs_pfb = np.array(f['PFB fs'])[0]
        qout = np.array(f['Qout'])[0]
xi = xi_adc
xq = xq_adc
x_adc_t = xi_adc + 1j*xq_adc
x_adc_t = np.array(x_adc_t, dtype=np.complex128)

In [ ]:
nsamp = 100
plt.figure(figsize=(10,4))
t = np.arange(len(x_adc_t))/Fs * 1e6
plt.plot(t[:nsamp],xi[:nsamp],label='real')
plt.plot(t[:nsamp],xq[:nsamp],label='imag')
plt.plot(t[:nsamp],np.abs(x_adc_t[:nsamp]),label='abs')
plt.xlabel("t [us]")
plt.legend(loc='upper right') 
plt.grid(True, alpha=0.3)

print(f"Mean value of real comp.: {np.mean(xi):.2f} LSB")
print(f"Mean value of imag comp.: {np.mean(xq):.2f} LSB")

In [ ]:
# x_adc_t = real_part + 1j * imag_part 
N = len(x_adc_t)

# --- 1. Window Processing ---
# A Hanning window reduces the tone's amplitude in the frequency domain.
# To recover the true amplitude, we divide by the Coherent Gain (0.5 for Hanning).
window = np.hanning(N)
x_windowed = x_adc_t * window

# --- 2. Complex FFT ---
# Standardize by N to reverse FFT processing gain.
# Note: For complex (I/Q) data, the FFT peak directly represents 
# the full amplitude of the complex phasor.
fft_complex = np.fft.fft(x_windowed) / N
fft_complex = np.fft.fftshift(fft_complex)
freqs = np.fft.fftshift(np.fft.fftfreq(N, 1/Fs))
    
# --- 3. Amplitude in dBFS ---
# Correct the linear magnitude using the Window Coherent Gain (0.5)
mag_linear = np.abs(fft_complex) / 0.5

# Convert to dBFS (decibels relative to Full Scale)
# We use our previously defined FS = 32768 (16-bit MSB-aligned)
with np.errstate(divide='ignore'):
    mag_dbfs = 20 * np.log10(mag_linear / FS)

# --- 4. PSD Calculation (dBFS/Hz) ---
# ENBW for Hanning is 1.5 bins. We convert this to Hz.
# PSD (dB/Hz) = Power_per_bin (dB) - 10*log10(Bin_Width_in_Hz * Window_Correction)
enbw_hz = 1.5 * (Fs / N) 
psd_dbfs_hz = mag_dbfs - 10 * np.log10(enbw_hz)

# --- 5. Plotting ---
plt.figure(figsize=(12, 8))

# Subplot 1: Amplitude Spectrum
plt.subplot(2, 1, 1)
plt.plot(freqs / 1e6, mag_dbfs, color='dodgerblue')
plt.title(f"Complex Spectrum (I/Q) | Fs = {Fs/1e6:.2f} MHz | {N}-pt FFT")
plt.ylabel("Amplitude [dBFS]")
plt.grid(True, alpha=0.4)
# Focus view: from peak down to 100dB below
plt.ylim([np.max(mag_dbfs)-100, 5]) 

# Peak Annotation
peak_idx = np.argmax(mag_dbfs)
plt.annotate(f'Peak: {mag_dbfs[peak_idx]:.2f} dBFS\n@{freqs[peak_idx]/1e6:.3f} MHz', 
             xy=(freqs[peak_idx]/1e6, mag_dbfs[peak_idx]), 
             xytext=(20, 5), textcoords='offset points',
             arrowprops=dict(arrowstyle='->', color='black'))

# Subplot 2: Power Spectral Density
plt.subplot(2, 1, 2)
plt.plot(freqs / 1e6, psd_dbfs_hz, color='coral')
plt.xlabel("Frequency [MHz]")
plt.ylabel("PSD [dBFS/Hz]")
plt.grid(True, alpha=0.4)

plt.tight_layout()
plt.show()
plt.savefig(f'D:\\Morgan\\20260206_cooldown\\data\\20260325\\images\\ADC_zcu111_AmpdBFS_and_PSD_1Tone_{f_tone/1e6:.2f}MHz_{Pin}dBm.png',dpi=300)

print(f"Calculated Tone: {mag_dbfs[peak_idx]:.2f} dBFS")
print(f"Average Noise Density: {np.median(psd_dbfs_hz):.2f} dBFS/Hz")
print(f"Peak linear amplitude: {mag_linear[peak_idx]:.2f}")
print(f"Maximum Magnitude in Time Data: {np.max(np.abs(x_adc_t))}")

In [ ]:
print("I mean:", np.mean(xi))
print("Q mean:", np.mean(xq))

print("I std:", np.std(xi))
print("Q std:", np.std(xq))

In [ ]:
plt.figure()
plt.plot(xi[:2000], xq[:2000], '.')
#plt.plot(xi, xq, '.')
plt.xlabel("I")
plt.ylabel("Q")
plt.title("IQ constellation")
plt.grid()

In [ ]:
#Amplitude variation measurement
r = np.abs(x_adc_t)
print("Amplitude std:", np.std(r))

#Circularity error
ratio = np.std(xi) / np.std(xq)
print("I/Q gain mismatch:", ratio)

#Module histogram
plt.figure()
plt.hist(np.abs(x_adc_t), bins=100)
plt.title("Amplitude distribution")
plt.show()

# PFB

## Fixed Qout

In [ ]:
# ==========================================
# SYSTEM PARAMETERS
# ==========================================
FS = 2**15  # ADC full scale (12-bit aligned to 16-bit)
tone_bins = 1

# RF chain parameters
K_adc = K # dB of attenuation BEFORE ADC

# ==========================================
# ADC CALIBRATION DERIVATION
# ==========================================
# We need to find the relationship between:
# - Input power to generator (Pin_gen in dBm)
# - Digital power measured in FFT (P_digital in dBFS)
#
# The chain is:
# Pin_gen → [-ANALOG_ATTEN] → Pin_adc → [ADC] → P_adc_digital → [PFB ÷64] → P_pfb_digital
#
# So: P_pfb_digital(dBFS) = P_adc_digital(dBFS) - PFB_SCALING
#
# We need to find: P_adc_digital(dBFS) as function of Pin_adc(dBm)
# This is the ADC calibration constant K_adc
#
# Pin_adc(dBm) = Pin_gen(dBm) - ANALOG_ATTEN
# P_adc_digital(dBFS) = Pin_adc(dBm) + K_adc
#
# Therefore:
# Pin_gen(dBm) = P_pfb_digital(dBFS) + PFB_SCALING - K_adc + ANALOG_ATTEN
# Pin_gen(dBm) = P_pfb_digital(dBFS) + (PFB_SCALING + ANALOG_ATTEN - K_adc)
#
# We want: Pin_gen = P_pfb_digital + CAL_CONSTANT
# So: CAL_CONSTANT = PFB_SCALING + ANALOG_ATTEN - K_adc
#
# From the data, we'll determine K_adc empirically

# ==========================================
# LOAD AND PROCESS DATA
# ==========================================
date = '20260327'
file_paths = [
    '00000_' + date + '_ADC+PFB_Cal-40dBm.h5',
    '00000_' + date + '_ADC+PFB_Cal-38dBm.h5',
    '00000_' + date + '_ADC+PFB_Cal-35dBm.h5',
    '00000_' + date + '_ADC+PFB_Cal-32dBm.h5',
    '00000_' + date + '_ADC+PFB_Cal-30dBm.h5',
    '00000_' + date + '_ADC+PFB_Cal-28dBm.h5',
    '00000_' + date + '_ADC+PFB_Cal-25dBm.h5',
    '00000_' + date + '_ADC+PFB_Cal-22dBm.h5',
    '00000_' + date + '_ADC+PFB_Cal-20dBm.h5',
    '00000_' + date + '_ADC+PFB_Cal-18dBm.h5',
    '00000_' + date + '_ADC+PFB_Cal-15dBm.h5',
    '00000_' + date + '_ADC+PFB_Cal-12dBm.h5',
    '00000_' + date + '_ADC+PFB_Cal-10dBm.h5',
    '00000_' + date + '_ADC+PFB_Cal-8dBm.h5',
    '00000_' + date + '_ADC+PFB_Cal-5dBm.h5',
    '00000_' + date + '_ADC+PFB_Cal0dBm.h5',
    '00000_' + date + '_ADC+PFB_Cal5dBm.h5',
    '00000_' + date + '_ADC+PFB_Cal6dBm.h5',
    '00000_' + date + '_ADC+PFB_Cal7dBm.h5',
    '00000_' + date + '_ADC+PFB_Cal8dBm.h5',
    '00000_' + date + '_ADC+PFB_Cal10dBm.h5',
]#
labels_list  = ['-40', '-38', '-35', '-32', '-30', '-28', '-25', '-22', '-20', '-18', '-15', '-12', '-10', '-8', '-5', '0', '+5', '+6', '+7', '+8', '+10']

# First pass: collect measurements
measurements = []

print("\n=== Processing Files ===")
for file in file_paths:
    m = re.search(r'(-?\+?\d+)dBm', file)
    Pin_gen = float(m.group(1))  # Generator input power
    with SlabFile('data/' + date + '/' + file, 'r') as f:
        xi_adc = np.array(f['xi_ADC'])[0]
        xq_adc = np.array(f['xq_ADC'])[0]
        xi_pfb = np.array(f['xi_PFB'])[0]
        xq_pfb = np.array(f['xq_PFB'])[0]
        fs_adc = np.array(f['ADC fs'])[0]
        fs_pfb = np.array(f['PFB fs'])[0]
        qout = np.array(f['Qout'])[0]

    Fs = fs_pfb*1e6
    PFB_QOUT = qout
    PFB_SCALING = 20*np.log10(2**PFB_QOUT)  # = 36.12 dB reduction

    if file == '00000_' + date + '_ADC+PFB_Cal-40dBm.h5':
        print("=== System Configuration ===")
        print(f"K_ADC: {K_adc} dB")
        print(f"PFB qout: {PFB_QOUT}")
        print(f"PFB scaling: {PFB_SCALING:.2f} dB (÷{2**PFB_QOUT})")
    
    x = xi_pfb + 1j*xq_pfb
    x = np.array(x, dtype=np.complex128)
    N = len(x)
    
    # Apply window
    window = np.hanning(N)
    coherent_gain = np.sum(window)/N
    xw = x * window
    
    # FFT
    fft_complex = np.fft.fftshift(np.fft.fft(xw)/N)
    freqs = np.fft.fftshift(np.fft.fftfreq(N, 1/Fs))
    mag = np.abs(fft_complex) / coherent_gain

    # plt.figure(dpi = 300)
    # plt.plot(freqs, mag, '.')
    # plt.title(file)

    # idx3 = np.where(freqs >=13000000.0)[0][0]
    # idx4 = np.where(freqs >= 14000000.0)[0][0]
    # print(mag)
    # print(mag[idx3:idx4])
    
    # Find tone and integrate power
    peak_idx = np.argmax(mag)
    idx0 = peak_idx - tone_bins
    idx1 = peak_idx + tone_bins + 1
    tone_power = np.sum(mag[idx0:idx1]**2)
    
    # Convert to dBFS (referenced to FS)
    P_pfb_digital = 10*np.log10(tone_power / (FS**2))
    
    measurements.append({
        'Pin_gen': Pin_gen,
        'P_pfb_digital': P_pfb_digital,
        'file': file
    })
    
    print(f"Pin={Pin_gen:+5.0f} dBm → PFB digital={P_pfb_digital:7.2f} dBFS @ Max Freq {freqs[peak_idx]}")

# ==========================================
# EMPIRICAL CALIBRATION
# ==========================================
print("\n=== Determining ADC Calibration ===")

# Linear regression to find K_adc
# We expect: Pin_gen = P_pfb_digital + CONSTANT
# Where: CONSTANT = PFB_SCALING + ANALOG_ATTEN - K_adc

# Use only linear region (exclude compressed points)
# Typically the first 7-8 points before compression
linear_measurements = measurements[:6]

Pin_gen_vals = np.array([m['Pin_gen'] for m in linear_measurements])
P_pfb_vals = np.array([m['P_pfb_digital'] for m in linear_measurements])

# Linear fit: Pin_gen = a * P_pfb + b
# We expect a ≈ 1 (slope should be 1 if no compression)
coeffs = np.polyfit(P_pfb_vals, Pin_gen_vals, 1)
slope = coeffs[0]
CONSTANT_measured = coeffs[1]

print(f"Linear fit: Pin_gen = {slope:.3f} * P_pfb + {CONSTANT_measured:.2f}")
print(f"Expected slope: 1.0 (actual: {slope:.3f})")

# From: CONSTANT = PFB_SCALING + ANALOG_ATTEN - K_adc
# We get: K_adc = PFB_SCALING + ANALOG_ATTEN - CONSTANT
ANALOG_ATTEN = -PFB_SCALING + K_adc + CONSTANT_measured

print(f"\nDerived Analog Attenuation (ANALOG_ATTEN): {ANALOG_ATTEN:.2f} dB")
print(f"This means: P_adc_digital(dBFS) = Pin_adc(dBm) + {K_adc:.2f}")

# Verify the chain
print(f"\n=== Verification ===")
print(f"PFB scaling: {PFB_SCALING:.2f} dB")
print(f"Analog atten: {ANALOG_ATTEN:.2f} dB")
print(f"ADC K_adc: {K_adc:.2f} dB")
print(f"Total calibration constant: {CONSTANT_measured:.2f} dB")
print(f"Expected: {PFB_SCALING + ANALOG_ATTEN - K_adc:.2f} dB ✓")

# ==========================================
# RECOMPUTE ALL POWERS WITH CALIBRATION
# ==========================================
CAL_CONSTANT = CONSTANT_measured

print(f"\n=== Recovering Input Powers ===")
print(f"Using: Pin_recovered = P_pfb_digital + {CAL_CONSTANT:.2f} dB")
print()
print(f"{'File':<45} {'Pin_true':>10} {'P_PFB':>12} {'Pin_recov':>12} {'Error':>10}")
print("="*95)

results = {
    'Pin_true': [],
    'Pin_recovered': [],
    'error': [],
    'P_pfb': []
}

for m in measurements:
    Pin_true = m['Pin_gen']
    P_pfb = m['P_pfb_digital']
    Pin_recovered = P_pfb + CAL_CONSTANT
    error = Pin_recovered - Pin_true
    
    results['Pin_true'].append(Pin_true)
    results['Pin_recovered'].append(Pin_recovered)
    results['error'].append(error)
    results['P_pfb'].append(P_pfb)

    filename = m['file'].split('/')[0]
    print(f"{filename:<45} {Pin_true:10.1f} {P_pfb:12.2f} {Pin_recovered:12.2f} {error:+10.2f}")

# ==========================================
# STATISTICS
# ==========================================
# Analyze only linear region
linear_errors = results['error'][:8]

print("\n" + "="*95)
print("=== Error Statistics (Linear Region) ===")
print(f"Mean error: {np.mean(linear_errors):+.3f} dB")
print(f"Std deviation: {np.std(linear_errors):.3f} dB")
print(f"Max |error|: {np.max(np.abs(linear_errors)):.3f} dB")
print(f"RMS error: {np.sqrt(np.mean(np.array(linear_errors)**2)):.3f} dB")

# ==========================================
# DETECT COMPRESSION
# ==========================================
print("\n=== Compression Analysis ===")
linear_mean = np.mean(results['Pin_recovered'][:7]) - np.mean(results['Pin_true'][:7])

for i in range(len(results['Pin_true'])):
    gain = results['Pin_recovered'][i] - results['Pin_true'][i]
    compression = linear_mean - gain
    
    if compression >= 1.0:
        print(f"1 dB compression at Pin = {results['Pin_true'][i]:+.1f} dBm")
        break
else:
    print("No compression detected in measured range")

# ==========================================
# PLOTTING
# ==========================================
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Recovered vs True Power
ax1 = axes[0, 0]
ax1.plot(results['Pin_true'], results['Pin_recovered'], 'o-', 
         markersize=8, linewidth=2, label='Recovered', color='blue')
ax1.plot(results['Pin_true'], results['Pin_true'], 'k--', 
         linewidth=2, label='Ideal (1:1)')
ax1.set_xlabel('True Input Power (dBm)', fontsize=12)
ax1.set_ylabel('Recovered Power (dBm)', fontsize=12)
ax1.set_title('Power Recovery Calibration', fontsize=13, fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# Add error annotations for points with >0.5dB error
for i in range(len(results['Pin_true'])):
    if abs(results['error'][i]) > 0.5:
        ax1.annotate(f'{results["error"][i]:+.1f}dB', 
                    xy=(results['Pin_true'][i], results['Pin_recovered'][i]),
                    xytext=(8, -8), textcoords='offset points',
                    fontsize=9, color='red',
                    arrowprops=dict(arrowstyle='->', color='red', lw=0.5))

# Plot 2: Error vs Input Power
ax2 = axes[0, 1]
ax2.plot(results['Pin_true'], results['error'], 'o-', 
         markersize=8, linewidth=2, color='green')
ax2.axhline(y=0, color='k', linestyle='--', linewidth=2)
ax2.axhline(y=0.5, color='orange', linestyle=':', linewidth=1.5, alpha=0.7)
ax2.axhline(y=-0.5, color='orange', linestyle=':', linewidth=1.5, alpha=0.7)
ax2.fill_between(results['Pin_true'], -0.5, 0.5, alpha=0.15, color='green')
ax2.set_xlabel('Input Power (dBm)', fontsize=12)
ax2.set_ylabel('Error (dB)', fontsize=12)
ax2.set_title('Calibration Error', fontsize=13, fontweight='bold')
ax2.grid(True, alpha=0.3)

# Plot 3: PFB Digital Level vs Input
ax3 = axes[1, 0]
ax3.plot(results['Pin_true'], results['P_pfb'], 'o-', 
         markersize=8, linewidth=2, color='purple')
ax3.set_xlabel('Input Power (dBm)', fontsize=12)
ax3.set_ylabel('PFB Digital Level (dBFS)', fontsize=12)
ax3.set_title('PFB Output vs Input Power', fontsize=13, fontweight='bold')
ax3.grid(True, alpha=0.3)

# Add expected line
expected_pfb = np.array(results['Pin_true']) - CAL_CONSTANT
ax3.plot(results['Pin_true'], expected_pfb, 'k--', 
         linewidth=2, alpha=0.5, label=f'Expected (slope=1)')
ax3.legend(fontsize=10)

# Plot 4: Error histogram
ax4 = axes[1, 1]
ax4.hist(linear_errors, bins=12, edgecolor='black', 
         alpha=0.7, color='green')
ax4.axvline(x=0, color='k', linestyle='--', linewidth=2, label='Zero error')
ax4.axvline(x=np.mean(linear_errors), color='r', linestyle='-', 
            linewidth=2, label=f'Mean={np.mean(linear_errors):+.3f} dB')
ax4.set_xlabel('Error (dB)', fontsize=12)
ax4.set_ylabel('Count', fontsize=12)
ax4.set_title('Error Distribution (Linear Region)', fontsize=13, fontweight='bold')
ax4.legend(fontsize=10)
ax4.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('images/' + date + '/power_calibration_final.pdf', dpi=300, bbox_inches='tight')
plt.show()

# ==========================================
# SUMMARY
# ==========================================
print("\n" + "="*95)
print("CALIBRATION SUMMARY")
print("="*95)
print(f"Calibration constant: {CAL_CONSTANT:.2f} dB")
print(f"  = PFB scaling ({PFB_SCALING:.2f}) + Analog atten ({ANALOG_ATTEN:.2f}) - ADC K_adc ({K_adc:.2f})")
print()
print("To recover input power from PFB data:")
print(f"  Pin (dBm) = P_PFB (dBFS) + {CAL_CONSTANT:.2f}")
print()
print(f"Calibration accuracy (linear region): ±{np.std(linear_errors):.3f} dB")
print("="*95)

In [ ]:
# ==========================================
# CALIBRATION PARAMETERS (from previous analysis)
# ==========================================
Fs = 256e6  # PFB channel sample rate
FS = 2**15  # ADC full scale
CAL_CONSTANT = -3.21  # dB (determined empirically)
tone_bins = 1

# Frequency of the tone WITHIN the PFB channel
# 300 MHz tone is at the center of the 256 MHz channel (bin 128)
# So the tone appears at: 300 MHz - 256 MHz = 44 MHz offset
f_tone_offset = 13.5*1e6  # Hz within the channel

print("=== Spectrum Analysis with Calibration ===")
print(f"Calibration constant: {CAL_CONSTANT:.2f} dB")
print(f"Expected tone frequency offset: {f_tone_offset/1e6:.1f} MHz")

# ==========================================
# LOAD AND PROCESS ALL FILES
# ==========================================
date = '20260327'
file_paths = [
    '00000_' + date + '_ADC+PFB_Cal-40dBm.h5',
    '00000_' + date + '_ADC+PFB_Cal-38dBm.h5',
    '00000_' + date + '_ADC+PFB_Cal-35dBm.h5',
    '00000_' + date + '_ADC+PFB_Cal-32dBm.h5',
    '00000_' + date + '_ADC+PFB_Cal-30dBm.h5',
    '00000_' + date + '_ADC+PFB_Cal-28dBm.h5',
    '00000_' + date + '_ADC+PFB_Cal-25dBm.h5',
    '00000_' + date + '_ADC+PFB_Cal-22dBm.h5',
    '00000_' + date + '_ADC+PFB_Cal-20dBm.h5',
    '00000_' + date + '_ADC+PFB_Cal-18dBm.h5',
    '00000_' + date + '_ADC+PFB_Cal-15dBm.h5',
    '00000_' + date + '_ADC+PFB_Cal-12dBm.h5',
    '00000_' + date + '_ADC+PFB_Cal-10dBm.h5',
    '00000_' + date + '_ADC+PFB_Cal-8dBm.h5',
    '00000_' + date + '_ADC+PFB_Cal-5dBm.h5',
    '00000_' + date + '_ADC+PFB_Cal0dBm.h5',
    '00000_' + date + '_ADC+PFB_Cal5dBm.h5',
    '00000_' + date + '_ADC+PFB_Cal6dBm.h5',
    '00000_' + date + '_ADC+PFB_Cal7dBm.h5',
    '00000_' + date + '_ADC+PFB_Cal8dBm.h5',
    '00000_' + date + '_ADC+PFB_Cal10dBm.h5',
]#

# Storage for results
all_results = []

for file in file_paths:
    m = re.search(r'(-?\+?\d+)dBm', file)
    Pin_gen = float(m.group(1))  # Generator input power
    with SlabFile('data/' + date + '/' + file, 'r') as f:
        xi_adc = np.array(f['xi_ADC'])[0]
        xq_adc = np.array(f['xq_ADC'])[0]
        xi_pfb = np.array(f['xi_PFB'])[0]
        xq_pfb = np.array(f['xq_PFB'])[0]
        fs_adc = np.array(f['ADC fs'])[0]
        fs_pfb = np.array(f['PFB fs'])[0]
        qout = np.array(f['Qout'])[0]
    
    x = xi_pfb + 1j*xq_pfb
    x = np.array(x, dtype=np.complex128)
    N = len(x)
    
    # Apply window
    window = np.hanning(N)
    coherent_gain = np.sum(window)/N
    xw = x * window
    
    # FFT
    fft_complex = np.fft.fftshift(np.fft.fft(xw)/N)
    freqs = np.fft.fftshift(np.fft.fftfreq(N, 1/Fs))
    mag = np.abs(fft_complex) / coherent_gain
    
    # Convert to dBFS
    mag_dbfs = 20*np.log10(mag / FS)
    
    # Convert to dBm using calibration
    mag_dbm = mag_dbfs + CAL_CONSTANT
    
    # Find peak (tone)
    peak_idx = np.argmax(mag)
    f_peak = freqs[peak_idx]
    
    # Integrate tone power
    idx0 = peak_idx - tone_bins
    idx1 = peak_idx + tone_bins + 1
    tone_power_linear = np.sum(mag[idx0:idx1]**2)
    tone_power_dbfs = 10*np.log10(tone_power_linear / (FS**2))
    tone_power_dbm = tone_power_dbfs + CAL_CONSTANT
    
    # Extract tone amplitude and phase
    tone_amplitude = mag[peak_idx]
    tone_amplitude_dbfs = 20*np.log10(tone_amplitude / FS)
    tone_amplitude_dbm = tone_amplitude_dbfs + CAL_CONSTANT
    
    tone_phase = np.angle(fft_complex[peak_idx])
    tone_phase_deg = np.degrees(tone_phase)
    
    all_results.append({
        'Pin': Pin_gen,
        'file': file,
        'freqs': freqs,
        'mag_dbm': mag_dbm,
        'mag_dbfs': mag_dbfs,
        'f_peak': f_peak,
        'tone_power_dbm': tone_power_dbm,
        'tone_amplitude': tone_amplitude,
        'tone_amplitude_dbm': tone_amplitude_dbm,
        'tone_phase': tone_phase,
        'tone_phase_deg': tone_phase_deg,
        'fft_complex': fft_complex,
        'peak_idx': peak_idx
     })

    print(f"Pin={Pin_gen:+5.0f} dBm: Peak at {f_peak/1e6:7.2f} MHz, "
          f"Tone={tone_power_dbm:7.2f} dBm, Phase={tone_phase_deg:+7.2f}°")

# ==========================================
# ANALYSIS: Amplitude and Phase vs Input Power
# ==========================================
Pin_vals = [r['Pin'] for r in all_results]
amplitude_vals = [r['tone_amplitude_dbm'] for r in all_results]
phase_vals = [r['tone_phase_deg'] for r in all_results]
freq_vals = [r['f_peak']/1e6 for r in all_results]  # MHz

print("\n=== Tone Characteristics ===")
print(f"{'Pin (dBm)':>10} {'Amplitude (dBm)':>18} {'Phase (deg)':>15} {'Freq (MHz)':>15}")
print("-"*65)
for i in range(len(Pin_vals)):
    print(f"{Pin_vals[i]:10.1f} {amplitude_vals[i]:18.2f} {phase_vals[i]:15.2f} {freq_vals[i]:15.3f}")

# Check if phase changes with power
phase_std = np.std(phase_vals)
phase_range = np.max(phase_vals) - np.min(phase_vals)
print(f"\nPhase statistics:")
print(f"  Mean: {np.mean(phase_vals):.2f}°")
print(f"  Std dev: {phase_std:.2f}°")
print(f"  Range: {phase_range:.2f}°")

if phase_std > 90:
    print("  !  Phase varies significantly with input power!")
else:
    print("  ✓ Phase is stable across power levels")

# ==========================================
# PLOTTING
# ==========================================
fig = plt.figure(figsize=(16, 12))
gs = fig.add_gridspec(3, 3, hspace=0.35, wspace=0.35)

# ==========================================
# Plot 1: Spectra overlay (dBm)
# ==========================================
ax1 = fig.add_subplot(gs[0, :2])

# Plot subset of spectra to avoid clutter
plot_indices = [0, len(all_results)//4, len(all_results)//2, 
                3*len(all_results)//4, len(all_results)-1]
colors = plt.cm.viridis(np.linspace(0, 1, len(plot_indices)))

for idx, color in zip(plot_indices, colors):
    r = all_results[idx]
    ax1.plot(r['freqs']/1e6, r['mag_dbm'], 
             label=f"Pin={r['Pin']:+.0f} dBm", 
             alpha=0.7, linewidth=1.5, color=color)

ax1.axvline(x=f_tone_offset/1e6, color='red', linestyle='--', 
            linewidth=2, alpha=0.5, label=f'Expected tone ({f_tone_offset/1e6:.0f} MHz)')
ax1.set_xlabel('Frequency (MHz)', fontsize=12)
ax1.set_ylabel('Power Spectral Density (dBm)', fontsize=12)
ax1.set_title('Calibrated Spectrum - Selected Input Powers', fontsize=13, fontweight='bold')
ax1.legend(fontsize=9, loc='upper right')
ax1.grid(True, alpha=0.3)
ax1.set_xlim([freqs[0]/1e6, freqs[-1]/1e6])

# ==========================================
# Plot 2: Waterfall plot (all spectra)
# ==========================================
ax2 = fig.add_subplot(gs[0, 2])

# Create 2D array for waterfall
spec_2d = np.array([r['mag_dbm'] for r in all_results])
extent = [freqs[0]/1e6, freqs[-1]/1e6, Pin_vals[0], Pin_vals[-1]]

im = ax2.imshow(spec_2d, aspect='auto', origin='lower', 
                extent=extent, cmap='viridis', 
                vmin=np.percentile(spec_2d, 5), 
                vmax=np.percentile(spec_2d, 95))
ax2.axvline(x=f_tone_offset/1e6, color='red', linestyle='--', 
            linewidth=2, alpha=0.7)
ax2.set_xlabel('Frequency (MHz)', fontsize=11)
ax2.set_ylabel('Input Power (dBm)', fontsize=11)
ax2.set_title('Waterfall Plot', fontsize=12, fontweight='bold')
cbar = plt.colorbar(im, ax=ax2)
cbar.set_label('Power (dBm)', fontsize=10)

# ==========================================
# Plot 3: Tone amplitude vs input power
# ==========================================
ax3 = fig.add_subplot(gs[1, 0])

ax3.plot(Pin_vals, amplitude_vals, 'o-', markersize=8, 
         linewidth=2, color='blue', label='Measured')
ax3.plot(Pin_vals, Pin_vals, 'k--', linewidth=2, 
         label='Ideal (1:1)')
ax3.set_xlabel('Input Power (dBm)', fontsize=12)
ax3.set_ylabel('Tone Amplitude (dBm)', fontsize=12)
ax3.set_title('Tone Amplitude Recovery', fontsize=12, fontweight='bold')
ax3.legend(fontsize=10)
ax3.grid(True, alpha=0.3)
print(f'Difference between Input Power and Tone Amplitude: {np.array(amplitude_vals) - np.array(Pin_vals)} dB')

# ==========================================
# Plot 4: Phase vs input power
# ==========================================
ax4 = fig.add_subplot(gs[1, 1])

ax4.plot(Pin_vals, phase_vals, 'o-', markersize=8, 
         linewidth=2, color='green')
ax4.axhline(y=np.mean(phase_vals), color='red', linestyle='--', 
            linewidth=2, alpha=0.7, label=f'Mean={np.mean(phase_vals):.1f}°')
ax4.fill_between(Pin_vals, 
                 np.mean(phase_vals) - phase_std,
                 np.mean(phase_vals) + phase_std,
                 alpha=0.2, color='green', label=f'±1σ ({phase_std:.2f}°)')
ax4.set_xlabel('Input Power (dBm)', fontsize=12)
ax4.set_ylabel('Tone Phase (degrees)', fontsize=12)
ax4.set_title('Tone Phase vs Input Power', fontsize=12, fontweight='bold')
ax4.legend(fontsize=10)
ax4.grid(True, alpha=0.3)

# ==========================================
# Plot 5: Frequency offset vs input power
# ==========================================
ax5 = fig.add_subplot(gs[1, 2])

ax5.plot(Pin_vals, freq_vals, 'o-', markersize=8, 
         linewidth=2, color='purple')
ax5.axhline(y=f_tone_offset/1e6, color='red', linestyle='--', 
            linewidth=2, alpha=0.7, label=f'Expected ({f_tone_offset/1e6:.1f} MHz)')
ax5.set_xlabel('Input Power (dBm)', fontsize=12)
ax5.set_ylabel('Peak Frequency (MHz)', fontsize=12)
ax5.set_title('Tone Frequency Stability', fontsize=12, fontweight='bold')
ax5.legend(fontsize=10)
ax5.grid(True, alpha=0.3)

# ==========================================
# Plot 6: Zoom on tone (highest power)
# ==========================================
ax6 = fig.add_subplot(gs[2, 0])

# Use highest input power for detail
highest_idx = np.argmax(Pin_vals)
r = all_results[highest_idx]

# Zoom around peak ±10 MHz
peak_freq_mhz = r['f_peak'] / 1e6
freq_mask = (r['freqs']/1e6 >= peak_freq_mhz - 10) & (r['freqs']/1e6 <= peak_freq_mhz + 10)

ax6.plot(r['freqs'][freq_mask]/1e6, r['mag_dbm'][freq_mask], 
         linewidth=2, color='blue')
ax6.axvline(x=peak_freq_mhz, color='red', linestyle='--', 
            linewidth=2, alpha=0.5, label=f'Peak at {peak_freq_mhz:.3f} MHz')
ax6.set_xlabel('Frequency (MHz)', fontsize=12)
ax6.set_ylabel('Power (dBm)', fontsize=12)
ax6.set_title(f'Tone Detail (Pin={r["Pin"]:+.0f} dBm)', 
              fontsize=12, fontweight='bold')
ax6.legend(fontsize=10)
ax6.grid(True, alpha=0.3)

# ==========================================
# Plot 7: IQ constellation (highest power)
# ==========================================
ax7 = fig.add_subplot(gs[2, 1])

# Time domain IQ plot for highest power
with SlabFile('data/' + date + '/' + all_results[highest_idx]['file'], 'r') as f:
    xi_adc = np.array(f['xi_ADC'])[0]
    xq_adc = np.array(f['xq_ADC'])[0]
    xi_pfb = np.array(f['xi_PFB'])[0]
    xq_pfb = np.array(f['xq_PFB'])[0]
    fs_adc = np.array(f['ADC fs'])[0]
    fs_pfb = np.array(f['PFB fs'])[0]
    qout = np.array(f['Qout'])[0]
    
x_highest = xi_adc + 1j*xq_adc
x_highest = np.array(x, dtype=np.complex128)
subsample = max(1, len(x_highest) // 2000)

ax7.scatter(np.real(x_highest[::subsample]), 
            np.imag(x_highest[::subsample]), 
            alpha=0.3, s=1, color='blue')
ax7.set_xlabel('I (Real)', fontsize=12)
ax7.set_ylabel('Q (Imaginary)', fontsize=12)
ax7.set_title(f'IQ Constellation (Pin={r["Pin"]:+.0f} dBm)', 
              fontsize=12, fontweight='bold')
ax7.grid(True, alpha=0.3)
ax7.axis('equal')

# Add circle showing expected amplitude
expected_radius = np.mean(np.abs(x_highest))
circle = plt.Circle((0, 0), expected_radius, fill=False, 
                    color='red', linewidth=2, linestyle='--', 
                    label=f'Mean |z|={expected_radius:.0f}')
ax7.add_patch(circle)
ax7.legend(fontsize=10)

# ==========================================
# Plot 8: Phase distribution (all powers)
# ==========================================
ax8 = fig.add_subplot(gs[2, 2])

ax8.hist(phase_vals, bins=15, edgecolor='black', 
         alpha=0.7, color='green')
ax8.axvline(x=np.mean(phase_vals), color='red', linestyle='--', 
            linewidth=2, label=f'Mean={np.mean(phase_vals):.2f}°')
ax8.set_xlabel('Phase (degrees)', fontsize=12)
ax8.set_ylabel('Count', fontsize=12)
ax8.set_title('Phase Distribution', fontsize=12, fontweight='bold')
ax8.legend(fontsize=10)
ax8.grid(True, alpha=0.3, axis='y')

plt.savefig('images/' + date + '/spectrum_analysis_complete.pdf', dpi=300, bbox_inches='tight')
plt.show()

# ==========================================
# DETAILED PHASE ANALYSIS
# ==========================================
print("\n" + "="*80)
print("PHASE ANALYSIS")
print("="*80)

# Unwrap phase if needed
phase_unwrapped = np.unwrap(np.array(phase_vals) * np.pi / 180) * 180 / np.pi

# Check for phase drift
phase_drift = phase_unwrapped[-1] - phase_unwrapped[0]
print(f"Phase drift across power range: {phase_drift:.2f}°")

# Check correlation between phase and power
correlation = np.corrcoef(Pin_vals, phase_vals)[0, 1]
print(f"Correlation between phase and input power: {correlation:.3f}")

if abs(correlation) > 0.5:
    print("  !  Strong correlation detected - phase depends on input power")
    
    # Linear fit
    coeffs = np.polyfit(Pin_vals, phase_vals, 1)
    print(f"  Phase ≈ {coeffs[1]:.2f}° + {coeffs[0]:.3f}° × Pin(dBm)")
else:
    print("  ✓ Phase is independent of input power (random variation)")

# ==========================================
# SUMMARY
# ==========================================
print("\n" + "="*80)
print("SUMMARY")
print("="*80)
print(f"Tone frequency: {np.mean(freq_vals):.3f} ± {np.std(freq_vals):.6f} MHz")
print(f"Tone phase: {np.mean(phase_vals):.2f}° ± {phase_std:.2f}°")
print(f"Phase stability: {'Good' if phase_std < 5 else 'Moderate' if phase_std < 15 else 'Poor'}")
print(f"Calibration constant: {CAL_CONSTANT:.2f} dB")
print("="*80)

## Vary Qout

In [ ]:
Qout_list = [0,1,2,3,4,5,6,7,8,9,10,11]
Cal_constant_list = []
# ==========================================
# SYSTEM PARAMETERS
# ==========================================
FS = 2**15  # ADC full scale (12-bit aligned to 16-bit)
tone_bins = 1

# RF chain parameters
K_adc = K # dB of attenuation BEFORE ADC

# ==========================================
# ADC CALIBRATION DERIVATION
# ==========================================
# We need to find the relationship between:
# - Input power to generator (Pin_gen in dBm)
# - Digital power measured in FFT (P_digital in dBFS)
#
# The chain is:
# Pin_gen → [-ANALOG_ATTEN] → Pin_adc → [ADC] → P_adc_digital → [PFB ÷64] → P_pfb_digital
#
# So: P_pfb_digital(dBFS) = P_adc_digital(dBFS) - PFB_SCALING
#
# We need to find: P_adc_digital(dBFS) as function of Pin_adc(dBm)
# This is the ADC calibration constant K_adc
#
# Pin_adc(dBm) = Pin_gen(dBm) - ANALOG_ATTEN
# P_adc_digital(dBFS) = Pin_adc(dBm) + K_adc
#
# Therefore:
# Pin_gen(dBm) = P_pfb_digital(dBFS) + PFB_SCALING - K_adc + ANALOG_ATTEN
# Pin_gen(dBm) = P_pfb_digital(dBFS) + (PFB_SCALING + ANALOG_ATTEN - K_adc)
#
# We want: Pin_gen = P_pfb_digital + CAL_CONSTANT
# So: CAL_CONSTANT = PFB_SCALING + ANALOG_ATTEN - K_adc
#
# From the data, we'll determine K_adc empirically

for ii, jj in enumerate(Qout_list):
    print(f'Qout: {jj}')

    if jj == 0:
        qq = 1
    else:
        qq = 0

    # ==========================================
    # LOAD AND PROCESS DATA
    # ==========================================
    date = '20260327'
    file_paths = [
        '0000' + str(qq) + '_' + date + '_ADC+PFB_Qout' + str(jj) + '_Cal-40dBm.h5',
        '0000' + str(qq) + '_' + date + '_ADC+PFB_Qout' + str(jj) + '_Cal-38dBm.h5',
        '0000' + str(qq) + '_' + date + '_ADC+PFB_Qout' + str(jj) + '_Cal-35dBm.h5',
        '0000' + str(qq) + '_' + date + '_ADC+PFB_Qout' + str(jj) + '_Cal-32dBm.h5',
        '0000' + str(qq) + '_' + date + '_ADC+PFB_Qout' + str(jj) + '_Cal-30dBm.h5',
        '0000' + str(qq) + '_' + date + '_ADC+PFB_Qout' + str(jj) + '_Cal-28dBm.h5',
        '0000' + str(qq) + '_' + date + '_ADC+PFB_Qout' + str(jj) + '_Cal-25dBm.h5',
        '0000' + str(qq) + '_' + date + '_ADC+PFB_Qout' + str(jj) + '_Cal-22dBm.h5',
        '0000' + str(qq) + '_' + date + '_ADC+PFB_Qout' + str(jj) + '_Cal-20dBm.h5',
        '0000' + str(qq) + '_' + date + '_ADC+PFB_Qout' + str(jj) + '_Cal-18dBm.h5',
        '0000' + str(qq) + '_' + date + '_ADC+PFB_Qout' + str(jj) + '_Cal-15dBm.h5',
        '0000' + str(qq) + '_' + date + '_ADC+PFB_Qout' + str(jj) + '_Cal-12dBm.h5',
        '0000' + str(qq) + '_' + date + '_ADC+PFB_Qout' + str(jj) + '_Cal-10dBm.h5',
        '0000' + str(qq) + '_' + date + '_ADC+PFB_Qout' + str(jj) + '_Cal-8dBm.h5',
        '0000' + str(qq) + '_' + date + '_ADC+PFB_Qout' + str(jj) + '_Cal-5dBm.h5',
        '0000' + str(qq) + '_' + date + '_ADC+PFB_Qout' + str(jj) + '_Cal0dBm.h5',
        '0000' + str(qq) + '_' + date + '_ADC+PFB_Qout' + str(jj) + '_Cal5dBm.h5',
        '0000' + str(qq) + '_' + date + '_ADC+PFB_Qout' + str(jj) + '_Cal6dBm.h5',
        '0000' + str(qq) + '_' + date + '_ADC+PFB_Qout' + str(jj) + '_Cal7dBm.h5',
        '0000' + str(qq) + '_' + date + '_ADC+PFB_Qout' + str(jj) + '_Cal8dBm.h5',
        '0000' + str(qq) + '_' + date + '_ADC+PFB_Qout' + str(jj) + '_Cal10dBm.h5',
    ]#
    labels_list  = ['-40', '-38', '-35', '-32', '-30', '-28', '-25', '-22', '-20', '-18', '-15', '-12', '-10', '-8', '-5', '0', '+5', '+6', '+7', '+8', '+10']
    
    # First pass: collect measurements
    measurements = []
    
    print("\n=== Processing Files ===")
    for file in file_paths:
        m = re.search(r'(-?\+?\d+)dBm', file)
        Pin_gen = float(m.group(1))  # Generator input power
        with SlabFile('D:\\Morgan\\20260206_cooldown\\data\\' + date + '\\' + file, 'r') as f:
            xi_adc = np.array(f['xi_ADC'])[0]
            xq_adc = np.array(f['xq_ADC'])[0]
            xi_pfb = np.array(f['xi_PFB'])[0]
            xq_pfb = np.array(f['xq_PFB'])[0]
            fs_adc = np.array(f['ADC fs'])[0]
            fs_pfb = np.array(f['PFB fs'])[0]
            qout = np.array(f['Qout'])[0]
    
        Fs = fs_pfb*1e6
        print(f'Expected Qout: {jj}, Saved Qout: {qout}')
        PFB_QOUT = qout
        PFB_SCALING = 20*np.log10(2**PFB_QOUT)  # = 36.12 dB reduction
    
        if file == '00000_' + date + '_ADC+PFB_Cal-40dBm.h5':
            print("=== System Configuration ===")
            print(f"K_ADC: {K_adc} dB")
            print(f"PFB qout: {PFB_QOUT}")
            print(f"PFB scaling: {PFB_SCALING:.2f} dB (÷{2**PFB_QOUT})")
        
        x = xi_pfb + 1j*xq_pfb
        x = np.array(x, dtype=np.complex128)
        N = len(x)
        
        # Apply window
        window = np.hanning(N)
        coherent_gain = np.sum(window)/N
        xw = x * window
        
        # FFT
        fft_complex = np.fft.fftshift(np.fft.fft(xw)/N)
        freqs = np.fft.fftshift(np.fft.fftfreq(N, 1/Fs))
        mag = np.abs(fft_complex) / coherent_gain
    
        # plt.figure(dpi = 300)
        # plt.plot(freqs, mag, '.')
        # plt.title(file)
    
        # idx3 = np.where(freqs >=13000000.0)[0][0]
        # idx4 = np.where(freqs >= 14000000.0)[0][0]
        # print(mag)
        # print(mag[idx3:idx4])
        
        # Find tone and integrate power
        peak_idx = np.argmax(mag)
        idx0 = peak_idx - tone_bins
        idx1 = peak_idx + tone_bins + 1
        tone_power = np.sum(mag[idx0:idx1]**2)
        
        # Convert to dBFS (referenced to FS)
        P_pfb_digital = 10*np.log10(tone_power / (FS**2))
        
        measurements.append({
            'Pin_gen': Pin_gen,
            'P_pfb_digital': P_pfb_digital,
            'file': file
        })
        
        print(f"Pin={Pin_gen:+5.0f} dBm → PFB digital={P_pfb_digital:7.2f} dBFS @ Max Freq {freqs[peak_idx]}")
    
    # ==========================================
    # EMPIRICAL CALIBRATION
    # ==========================================
    print("\n=== Determining ADC Calibration ===")
    
    # Linear regression to find K_adc
    # We expect: Pin_gen = P_pfb_digital + CONSTANT
    # Where: CONSTANT = PFB_SCALING + ANALOG_ATTEN - K_adc
    
    # Use only linear region (exclude compressed points)
    # Typically the first 7-8 points before compression
    linear_measurements = measurements[:6]
    
    Pin_gen_vals = np.array([m['Pin_gen'] for m in linear_measurements])
    P_pfb_vals = np.array([m['P_pfb_digital'] for m in linear_measurements])

    # Linear fit: Pin_gen = a * P_pfb + b
    # We expect a ≈ 1 (slope should be 1 if no compression)
    coeffs = np.polyfit(P_pfb_vals, Pin_gen_vals, 1)
    slope = coeffs[0]
    CONSTANT_measured = coeffs[1]
    
    print(f"Linear fit: Pin_gen = {slope:.3f} * P_pfb + {CONSTANT_measured:.2f}")
    print(f"Expected slope: 1.0 (actual: {slope:.3f})")
    
    # From: CONSTANT = PFB_SCALING + ANALOG_ATTEN - K_adc
    # We get: K_adc = PFB_SCALING + ANALOG_ATTEN - CONSTANT
    ANALOG_ATTEN = -PFB_SCALING + K_adc + CONSTANT_measured
    
    print(f"\nDerived Analog Attenuation (ANALOG_ATTEN): {ANALOG_ATTEN:.2f} dB")
    print(f"This means: P_adc_digital(dBFS) = Pin_adc(dBm) + {K_adc:.2f}")
    
    # Verify the chain
    print(f"\n=== Verification ===")
    print(f"PFB scaling: {PFB_SCALING:.2f} dB")
    print(f"Analog atten: {ANALOG_ATTEN:.2f} dB")
    print(f"ADC K_adc: {K_adc:.2f} dB")
    print(f"Total calibration constant: {CONSTANT_measured:.2f} dB")
    print(f"Expected: {PFB_SCALING + ANALOG_ATTEN - K_adc:.2f} dB ✓")
    
    # ==========================================
    # RECOMPUTE ALL POWERS WITH CALIBRATION
    # ==========================================
    CAL_CONSTANT = CONSTANT_measured
    Cal_constant_list.append(CAL_CONSTANT)
    
    print(f"\n=== Recovering Input Powers ===")
    print(f"Using: Pin_recovered = P_pfb_digital + {CAL_CONSTANT:.2f} dB")
    print()
    print(f"{'File':<45} {'Pin_true':>10} {'P_PFB':>12} {'Pin_recov':>12} {'Error':>10}")
    print("="*95)
    
    results = {
        'Pin_true': [],
        'Pin_recovered': [],
        'error': [],
        'P_pfb': []
    }
    
    for m in measurements:
        Pin_true = m['Pin_gen']
        P_pfb = m['P_pfb_digital']
        Pin_recovered = P_pfb + CAL_CONSTANT
        error = Pin_recovered - Pin_true
        
        results['Pin_true'].append(Pin_true)
        results['Pin_recovered'].append(Pin_recovered)
        results['error'].append(error)
        results['P_pfb'].append(P_pfb)
    
        filename = m['file'].split('/')[0]
        print(f"{filename:<45} {Pin_true:10.1f} {P_pfb:12.2f} {Pin_recovered:12.2f} {error:+10.2f}")
    
    # ==========================================
    # STATISTICS
    # ==========================================
    # Analyze only linear region
    linear_errors = results['error'][:8]
    
    print("\n" + "="*95)
    print("=== Error Statistics (Linear Region) ===")
    print(f"Mean error: {np.mean(linear_errors):+.3f} dB")
    print(f"Std deviation: {np.std(linear_errors):.3f} dB")
    print(f"Max |error|: {np.max(np.abs(linear_errors)):.3f} dB")
    print(f"RMS error: {np.sqrt(np.mean(np.array(linear_errors)**2)):.3f} dB")

    # ==========================================
    # DETECT COMPRESSION
    # ==========================================
    print("\n=== Compression Analysis ===")
    linear_mean = np.mean(results['Pin_recovered'][:7]) - np.mean(results['Pin_true'][:7])
    
    for i in range(len(results['Pin_true'])):
        gain = results['Pin_recovered'][i] - results['Pin_true'][i]
        compression = linear_mean - gain
        
        if compression >= 1.0:
            print(f"1 dB compression at Pin = {results['Pin_true'][i]:+.1f} dBm")
            break
    else:
        print("No compression detected in measured range")
    
    # ==========================================
    # PLOTTING
    # ==========================================
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Plot 1: Recovered vs True Power
    ax1 = axes[0, 0]
    ax1.plot(results['Pin_true'], results['Pin_recovered'], 'o-', 
             markersize=8, linewidth=2, label='Recovered', color='blue')
    ax1.plot(results['Pin_true'], results['Pin_true'], 'k--', 
             linewidth=2, label='Ideal (1:1)')
    ax1.set_xlabel('True Input Power (dBm)', fontsize=12)
    ax1.set_ylabel('Recovered Power (dBm)', fontsize=12)
    ax1.set_title('Power Recovery Calibration', fontsize=13, fontweight='bold')
    ax1.legend(fontsize=11)
    ax1.grid(True, alpha=0.3)
    
    # Add error annotations for points with >0.5dB error
    for i in range(len(results['Pin_true'])):
        if abs(results['error'][i]) > 0.5:
            ax1.annotate(f'{results["error"][i]:+.1f}dB', 
                        xy=(results['Pin_true'][i], results['Pin_recovered'][i]),
                        xytext=(8, -8), textcoords='offset points',
                        fontsize=9, color='red',
                        arrowprops=dict(arrowstyle='->', color='red', lw=0.5))
    
    # Plot 2: Error vs Input Power
    ax2 = axes[0, 1]
    ax2.plot(results['Pin_true'], results['error'], 'o-', 
             markersize=8, linewidth=2, color='green')
    ax2.axhline(y=0, color='k', linestyle='--', linewidth=2)
    ax2.axhline(y=0.5, color='orange', linestyle=':', linewidth=1.5, alpha=0.7)
    ax2.axhline(y=-0.5, color='orange', linestyle=':', linewidth=1.5, alpha=0.7)
    ax2.fill_between(results['Pin_true'], -0.5, 0.5, alpha=0.15, color='green')
    ax2.set_xlabel('Input Power (dBm)', fontsize=12)
    ax2.set_ylabel('Error (dB)', fontsize=12)
    ax2.set_title('Calibration Error', fontsize=13, fontweight='bold')
    ax2.grid(True, alpha=0.3)
    
    # Plot 3: PFB Digital Level vs Input
    ax3 = axes[1, 0]
    ax3.plot(results['Pin_true'], results['P_pfb'], 'o-', 
             markersize=8, linewidth=2, color='purple')
    ax3.set_xlabel('Input Power (dBm)', fontsize=12)
    ax3.set_ylabel('PFB Digital Level (dBFS)', fontsize=12)
    ax3.set_title('PFB Output vs Input Power', fontsize=13, fontweight='bold')
    ax3.grid(True, alpha=0.3)
    
    # Add expected line
    expected_pfb = np.array(results['Pin_true']) - CAL_CONSTANT
    ax3.plot(results['Pin_true'], expected_pfb, 'k--', 
             linewidth=2, alpha=0.5, label=f'Expected (slope=1)')
    ax3.legend(fontsize=10)
    
    # Plot 4: Error histogram
    ax4 = axes[1, 1]
    ax4.hist(linear_errors, bins=12, edgecolor='black', 
             alpha=0.7, color='green')
    ax4.axvline(x=0, color='k', linestyle='--', linewidth=2, label='Zero error')
    ax4.axvline(x=np.mean(linear_errors), color='r', linestyle='-', 
                linewidth=2, label=f'Mean={np.mean(linear_errors):+.3f} dB')
    ax4.set_xlabel('Error (dB)', fontsize=12)
    ax4.set_ylabel('Count', fontsize=12)
    ax4.set_title('Error Distribution (Linear Region)', fontsize=13, fontweight='bold')
    ax4.legend(fontsize=10)
    ax4.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.savefig('D:\\Morgan\\20260206_cooldown\\data\\' + date + '\\images\\power_calibration_final.pdf', dpi=300, bbox_inches='tight')
    plt.show()
    
    # ==========================================
    # SUMMARY
    # ==========================================
    print("\n" + "="*95)
    print("CALIBRATION SUMMARY")
    print("="*95)
    print(f"Calibration constant: {CAL_CONSTANT:.2f} dB")
    print(f"  = PFB scaling ({PFB_SCALING:.2f}) + Analog atten ({ANALOG_ATTEN:.2f}) - ADC K_adc ({K_adc:.2f})")
    print()
    print("To recover input power from PFB data:")
    print(f"  Pin (dBm) = P_PFB (dBFS) + {CAL_CONSTANT:.2f}")
    print()
    print(f"Calibration accuracy (linear region): ±{np.std(linear_errors):.3f} dB")
    print("="*95)
print(Cal_constant_list)

In [ ]:
Qout_list = [0,1,2,3,4,5,6,7,8,9,10,11]
for ii, jj in enumerate(Qout_list):
    print(f'Qout: {jj}')
    # ==========================================
    # CALIBRATION PARAMETERS (from previous analysis)
    # ==========================================
    Fs = 256e6  # PFB channel sample rate
    FS = 2**15  # ADC full scale
    CAL_CONSTANT = Cal_constant_list[ii]  # dB (determined empirically)
    tone_bins = 1
    
    # Frequency of the tone WITHIN the PFB channel
    # 300 MHz tone is at the center of the 256 MHz channel (bin 128)
    # So the tone appears at: 300 MHz - 256 MHz = 44 MHz offset
    f_tone_offset = 13.5*1e6  # Hz within the channel
    
    print("=== Spectrum Analysis with Calibration ===")
    print(f"Calibration constant: {CAL_CONSTANT:.2f} dB")
    print(f"Expected tone frequency offset: {f_tone_offset/1e6:.1f} MHz")
    
    # ==========================================
    # LOAD AND PROCESS ALL FILES
    # ==========================================
    date = '20260327'
    file_paths = [
        '0000' + str(qq) + '_' + date + '_ADC+PFB_Qout' + str(jj) + '_Cal-40dBm.h5',
        '0000' + str(qq) + '_' + date + '_ADC+PFB_Qout' + str(jj) + '_Cal-38dBm.h5',
        '0000' + str(qq) + '_' + date + '_ADC+PFB_Qout' + str(jj) + '_Cal-35dBm.h5',
        '0000' + str(qq) + '_' + date + '_ADC+PFB_Qout' + str(jj) + '_Cal-32dBm.h5',
        '0000' + str(qq) + '_' + date + '_ADC+PFB_Qout' + str(jj) + '_Cal-30dBm.h5',
        '0000' + str(qq) + '_' + date + '_ADC+PFB_Qout' + str(jj) + '_Cal-28dBm.h5',
        '0000' + str(qq) + '_' + date + '_ADC+PFB_Qout' + str(jj) + '_Cal-25dBm.h5',
        '0000' + str(qq) + '_' + date + '_ADC+PFB_Qout' + str(jj) + '_Cal-22dBm.h5',
        '0000' + str(qq) + '_' + date + '_ADC+PFB_Qout' + str(jj) + '_Cal-20dBm.h5',
        '0000' + str(qq) + '_' + date + '_ADC+PFB_Qout' + str(jj) + '_Cal-18dBm.h5',
        '0000' + str(qq) + '_' + date + '_ADC+PFB_Qout' + str(jj) + '_Cal-15dBm.h5',
        '0000' + str(qq) + '_' + date + '_ADC+PFB_Qout' + str(jj) + '_Cal-12dBm.h5',
        '0000' + str(qq) + '_' + date + '_ADC+PFB_Qout' + str(jj) + '_Cal-10dBm.h5',
        '0000' + str(qq) + '_' + date + '_ADC+PFB_Qout' + str(jj) + '_Cal-8dBm.h5',
        '0000' + str(qq) + '_' + date + '_ADC+PFB_Qout' + str(jj) + '_Cal-5dBm.h5',
        '0000' + str(qq) + '_' + date + '_ADC+PFB_Qout' + str(jj) + '_Cal0dBm.h5',
        '0000' + str(qq) + '_' + date + '_ADC+PFB_Qout' + str(jj) + '_Cal5dBm.h5',
        '0000' + str(qq) + '_' + date + '_ADC+PFB_Qout' + str(jj) + '_Cal6dBm.h5',
        '0000' + str(qq) + '_' + date + '_ADC+PFB_Qout' + str(jj) + '_Cal7dBm.h5',
        '0000' + str(qq) + '_' + date + '_ADC+PFB_Qout' + str(jj) + '_Cal8dBm.h5',
        '0000' + str(qq) + '_' + date + '_ADC+PFB_Qout' + str(jj) + '_Cal10dBm.h5',
    ]#
    
    # Storage for results
    all_results = []
    
    for file in file_paths:
        m = re.search(r'(-?\+?\d+)dBm', file)
        Pin_gen = float(m.group(1))  # Generator input power
        with SlabFile('D:\\Morgan\\20260206_cooldown\\data\\' + date + '\\' + file, 'r') as f:
            xi_adc = np.array(f['xi_ADC'])[0]
            xq_adc = np.array(f['xq_ADC'])[0]
            xi_pfb = np.array(f['xi_PFB'])[0]
            xq_pfb = np.array(f['xq_PFB'])[0]
            fs_adc = np.array(f['ADC fs'])[0]
            fs_pfb = np.array(f['PFB fs'])[0]
            qout = np.array(f['Qout'])[0]
        
        x = xi_pfb + 1j*xq_pfb
        x = np.array(x, dtype=np.complex128)
        N = len(x)
        
        # Apply window
        window = np.hanning(N)
        coherent_gain = np.sum(window)/N
        xw = x * window
        
        # FFT
        fft_complex = np.fft.fftshift(np.fft.fft(xw)/N)
        freqs = np.fft.fftshift(np.fft.fftfreq(N, 1/Fs))
        mag = np.abs(fft_complex) / coherent_gain
        
        # Convert to dBFS
        mag_dbfs = 20*np.log10(mag / FS)
        
        # Convert to dBm using calibration
        mag_dbm = mag_dbfs + CAL_CONSTANT
        
        # Find peak (tone)
        peak_idx = np.argmax(mag)
        f_peak = freqs[peak_idx]
        
        # Integrate tone power
        idx0 = peak_idx - tone_bins
        idx1 = peak_idx + tone_bins + 1
        tone_power_linear = np.sum(mag[idx0:idx1]**2)
        tone_power_dbfs = 10*np.log10(tone_power_linear / (FS**2))
        tone_power_dbm = tone_power_dbfs + CAL_CONSTANT
        
        # Extract tone amplitude and phase
        tone_amplitude = mag[peak_idx]
        tone_amplitude_dbfs = 20*np.log10(tone_amplitude / FS)
        tone_amplitude_dbm = tone_amplitude_dbfs + CAL_CONSTANT
        
        tone_phase = np.angle(fft_complex[peak_idx])
        tone_phase_deg = np.degrees(tone_phase)
        
        all_results.append({
            'Pin': Pin_gen,
            'file': file,
            'freqs': freqs,
            'mag_dbm': mag_dbm,
            'mag_dbfs': mag_dbfs,
            'f_peak': f_peak,
            'tone_power_dbm': tone_power_dbm,
            'tone_amplitude': tone_amplitude,
            'tone_amplitude_dbm': tone_amplitude_dbm,
            'tone_phase': tone_phase,
            'tone_phase_deg': tone_phase_deg,
            'fft_complex': fft_complex,
            'peak_idx': peak_idx
         })
    
        print(f"Pin={Pin_gen:+5.0f} dBm: Peak at {f_peak/1e6:7.2f} MHz, "
              f"Tone={tone_power_dbm:7.2f} dBm, Phase={tone_phase_deg:+7.2f}°")
    
    # ==========================================
    # ANALYSIS: Amplitude and Phase vs Input Power
    # ==========================================
    Pin_vals = [r['Pin'] for r in all_results]
    amplitude_vals = [r['tone_amplitude_dbm'] for r in all_results]
    phase_vals = [r['tone_phase_deg'] for r in all_results]
    freq_vals = [r['f_peak']/1e6 for r in all_results]  # MHz
    
    print("\n=== Tone Characteristics ===")
    print(f"{'Pin (dBm)':>10} {'Amplitude (dBm)':>18} {'Phase (deg)':>15} {'Freq (MHz)':>15}")
    print("-"*65)
    for i in range(len(Pin_vals)):
        print(f"{Pin_vals[i]:10.1f} {amplitude_vals[i]:18.2f} {phase_vals[i]:15.2f} {freq_vals[i]:15.3f}")
    
    # Check if phase changes with power
    phase_std = np.std(phase_vals)
    phase_range = np.max(phase_vals) - np.min(phase_vals)
    print(f"\nPhase statistics:")
    print(f"  Mean: {np.mean(phase_vals):.2f}°")
    print(f"  Std dev: {phase_std:.2f}°")
    print(f"  Range: {phase_range:.2f}°")
    
    if phase_std > 90:
        print("  !  Phase varies significantly with input power!")
    else:
        print("  ✓ Phase is stable across power levels")
    
    # ==========================================
    # PLOTTING
    # ==========================================
    fig = plt.figure(figsize=(16, 12))
    gs = fig.add_gridspec(3, 3, hspace=0.35, wspace=0.35)
    
    # ==========================================
    # Plot 1: Spectra overlay (dBm)
    # ==========================================
    ax1 = fig.add_subplot(gs[0, :2])
    
    # Plot subset of spectra to avoid clutter
    plot_indices = [0, len(all_results)//4, len(all_results)//2, 
                    3*len(all_results)//4, len(all_results)-1]
    colors = plt.cm.viridis(np.linspace(0, 1, len(plot_indices)))
    
    for idx, color in zip(plot_indices, colors):
        r = all_results[idx]
        ax1.plot(r['freqs']/1e6, r['mag_dbm'], 
                 label=f"Pin={r['Pin']:+.0f} dBm", 
                 alpha=0.7, linewidth=1.5, color=color)
    
    ax1.axvline(x=f_tone_offset/1e6, color='red', linestyle='--', 
                linewidth=2, alpha=0.5, label=f'Expected tone ({f_tone_offset/1e6:.0f} MHz)')
    ax1.set_xlabel('Frequency (MHz)', fontsize=12)
    ax1.set_ylabel('Power Spectral Density (dBm)', fontsize=12)
    ax1.set_title('Calibrated Spectrum - Selected Input Powers', fontsize=13, fontweight='bold')
    ax1.legend(fontsize=9, loc='upper right')
    ax1.grid(True, alpha=0.3)
    ax1.set_xlim([freqs[0]/1e6, freqs[-1]/1e6])
    
    # ==========================================
    # Plot 2: Waterfall plot (all spectra)
    # ==========================================
    ax2 = fig.add_subplot(gs[0, 2])
    
    # Create 2D array for waterfall
    spec_2d = np.array([r['mag_dbm'] for r in all_results])
    extent = [freqs[0]/1e6, freqs[-1]/1e6, Pin_vals[0], Pin_vals[-1]]
    
    im = ax2.imshow(spec_2d, aspect='auto', origin='lower', 
                    extent=extent, cmap='viridis', 
                    vmin=np.percentile(spec_2d, 5), 
                    vmax=np.percentile(spec_2d, 95))
    ax2.axvline(x=f_tone_offset/1e6, color='red', linestyle='--', 
                linewidth=2, alpha=0.7)
    ax2.set_xlabel('Frequency (MHz)', fontsize=11)
    ax2.set_ylabel('Input Power (dBm)', fontsize=11)
    ax2.set_title('Waterfall Plot', fontsize=12, fontweight='bold')
    cbar = plt.colorbar(im, ax=ax2)
    cbar.set_label('Power (dBm)', fontsize=10)
    
    # ==========================================
    # Plot 3: Tone amplitude vs input power
    # ==========================================
    ax3 = fig.add_subplot(gs[1, 0])
    
    ax3.plot(Pin_vals, amplitude_vals, 'o-', markersize=8, 
             linewidth=2, color='blue', label='Measured')
    ax3.plot(Pin_vals, Pin_vals, 'k--', linewidth=2, 
             label='Ideal (1:1)')
    ax3.set_xlabel('Input Power (dBm)', fontsize=12)
    ax3.set_ylabel('Tone Amplitude (dBm)', fontsize=12)
    ax3.set_title('Tone Amplitude Recovery', fontsize=12, fontweight='bold')
    ax3.legend(fontsize=10)
    ax3.grid(True, alpha=0.3)
    print(f'Difference between Input Power and Tone Amplitude: {np.array(amplitude_vals) - np.array(Pin_vals)} dB')
    
    # ==========================================
    # Plot 4: Phase vs input power
    # ==========================================
    ax4 = fig.add_subplot(gs[1, 1])
    
    ax4.plot(Pin_vals, phase_vals, 'o-', markersize=8, 
             linewidth=2, color='green')
    ax4.axhline(y=np.mean(phase_vals), color='red', linestyle='--', 
                linewidth=2, alpha=0.7, label=f'Mean={np.mean(phase_vals):.1f}°')
    ax4.fill_between(Pin_vals, 
                     np.mean(phase_vals) - phase_std,
                     np.mean(phase_vals) + phase_std,
                     alpha=0.2, color='green', label=f'±1σ ({phase_std:.2f}°)')
    ax4.set_xlabel('Input Power (dBm)', fontsize=12)
    ax4.set_ylabel('Tone Phase (degrees)', fontsize=12)
    ax4.set_title('Tone Phase vs Input Power', fontsize=12, fontweight='bold')
    ax4.legend(fontsize=10)
    ax4.grid(True, alpha=0.3)
    
    # ==========================================
    # Plot 5: Frequency offset vs input power
    # ==========================================
    ax5 = fig.add_subplot(gs[1, 2])
    
    ax5.plot(Pin_vals, freq_vals, 'o-', markersize=8, 
             linewidth=2, color='purple')
    ax5.axhline(y=f_tone_offset/1e6, color='red', linestyle='--', 
                linewidth=2, alpha=0.7, label=f'Expected ({f_tone_offset/1e6:.1f} MHz)')
    ax5.set_xlabel('Input Power (dBm)', fontsize=12)
    ax5.set_ylabel('Peak Frequency (MHz)', fontsize=12)
    ax5.set_title('Tone Frequency Stability', fontsize=12, fontweight='bold')
    ax5.legend(fontsize=10)
    ax5.grid(True, alpha=0.3)
    
    # ==========================================
    # Plot 6: Zoom on tone (highest power)
    # ==========================================
    ax6 = fig.add_subplot(gs[2, 0])
    
    # Use highest input power for detail
    highest_idx = np.argmax(Pin_vals)
    r = all_results[highest_idx]
    
    # Zoom around peak ±10 MHz
    peak_freq_mhz = r['f_peak'] / 1e6
    freq_mask = (r['freqs']/1e6 >= peak_freq_mhz - 10) & (r['freqs']/1e6 <= peak_freq_mhz + 10)
    
    ax6.plot(r['freqs'][freq_mask]/1e6, r['mag_dbm'][freq_mask], 
             linewidth=2, color='blue')
    ax6.axvline(x=peak_freq_mhz, color='red', linestyle='--', 
                linewidth=2, alpha=0.5, label=f'Peak at {peak_freq_mhz:.3f} MHz')
    ax6.set_xlabel('Frequency (MHz)', fontsize=12)
    ax6.set_ylabel('Power (dBm)', fontsize=12)
    ax6.set_title(f'Tone Detail (Pin={r["Pin"]:+.0f} dBm)', 
                  fontsize=12, fontweight='bold')
    ax6.legend(fontsize=10)
    ax6.grid(True, alpha=0.3)
    
    # ==========================================
    # Plot 7: IQ constellation (highest power)
    # ==========================================
    ax7 = fig.add_subplot(gs[2, 1])
    
    # Time domain IQ plot for highest power
    with SlabFile('D:\\Morgan\\20260206_cooldown\\data\\' + date + '\\' + all_results[highest_idx]['file'], 'r') as f:
        xi_adc = np.array(f['xi_ADC'])[0]
        xq_adc = np.array(f['xq_ADC'])[0]
        xi_pfb = np.array(f['xi_PFB'])[0]
        xq_pfb = np.array(f['xq_PFB'])[0]
        fs_adc = np.array(f['ADC fs'])[0]
        fs_pfb = np.array(f['PFB fs'])[0]
        qout = np.array(f['Qout'])[0]
        
    x_highest = xi_adc + 1j*xq_adc
    x_highest = np.array(x, dtype=np.complex128)
    subsample = max(1, len(x_highest) // 2000)
    
    ax7.scatter(np.real(x_highest[::subsample]), 
                np.imag(x_highest[::subsample]), 
                alpha=0.3, s=1, color='blue')
    ax7.set_xlabel('I (Real)', fontsize=12)
    ax7.set_ylabel('Q (Imaginary)', fontsize=12)
    ax7.set_title(f'IQ Constellation (Pin={r["Pin"]:+.0f} dBm)', 
                  fontsize=12, fontweight='bold')
    ax7.grid(True, alpha=0.3)
    ax7.axis('equal')
    
    # Add circle showing expected amplitude
    expected_radius = np.mean(np.abs(x_highest))
    circle = plt.Circle((0, 0), expected_radius, fill=False, 
                        color='red', linewidth=2, linestyle='--', 
                        label=f'Mean |z|={expected_radius:.0f}')
    ax7.add_patch(circle)
    ax7.legend(fontsize=10)
    
    # ==========================================
    # Plot 8: Phase distribution (all powers)
    # ==========================================
    ax8 = fig.add_subplot(gs[2, 2])
    
    ax8.hist(phase_vals, bins=15, edgecolor='black', 
             alpha=0.7, color='green')
    ax8.axvline(x=np.mean(phase_vals), color='red', linestyle='--', 
                linewidth=2, label=f'Mean={np.mean(phase_vals):.2f}°')
    ax8.set_xlabel('Phase (degrees)', fontsize=12)
    ax8.set_ylabel('Count', fontsize=12)
    ax8.set_title('Phase Distribution', fontsize=12, fontweight='bold')
    ax8.legend(fontsize=10)
    ax8.grid(True, alpha=0.3, axis='y')
    
    plt.savefig('D:\\Morgan\\20260206_cooldown\\data\\' + date + '\\spectrum_analysis_complete.pdf', dpi=300, bbox_inches='tight')
    plt.show()
    
    # ==========================================
    # DETAILED PHASE ANALYSIS
    # ==========================================
    print("\n" + "="*80)
    print("PHASE ANALYSIS")
    print("="*80)
    
    # Unwrap phase if needed
    phase_unwrapped = np.unwrap(np.array(phase_vals) * np.pi / 180) * 180 / np.pi
    
    # Check for phase drift
    phase_drift = phase_unwrapped[-1] - phase_unwrapped[0]
    print(f"Phase drift across power range: {phase_drift:.2f}°")
    
    # Check correlation between phase and power
    correlation = np.corrcoef(Pin_vals, phase_vals)[0, 1]
    print(f"Correlation between phase and input power: {correlation:.3f}")
    
    if abs(correlation) > 0.5:
        print("  !  Strong correlation detected - phase depends on input power")
        
        # Linear fit
        coeffs = np.polyfit(Pin_vals, phase_vals, 1)
        print(f"  Phase ≈ {coeffs[1]:.2f}° + {coeffs[0]:.3f}° × Pin(dBm)")
    else:
        print("  ✓ Phase is independent of input power (random variation)")
    
    # ==========================================
    # SUMMARY
    # ==========================================
    print("\n" + "="*80)
    print("SUMMARY")
    print("="*80)
    print(f"Tone frequency: {np.mean(freq_vals):.3f} ± {np.std(freq_vals):.6f} MHz")
    print(f"Tone phase: {np.mean(phase_vals):.2f}° ± {phase_std:.2f}°")
    print(f"Phase stability: {'Good' if phase_std < 5 else 'Moderate' if phase_std < 15 else 'Poor'}")
    print(f"Calibration constant: {CAL_CONSTANT:.2f} dB")
    print("="*80)

# DDS+CIC

In [ ]:
# ================================
# User parameters (update these)
# ================================
data_file = 'D:\\Morgan\\20260206_cooldown\\data\\' + date + '\\00000_' + date + '_DDS+CIC_Cal-12dBm.h5'  # path to your saved data
fs_ch = 256e6       # ADC sampling frequency before decimation (Hz)
N = 3               # Number of CIC stages (3rd order CIC as IP block)
# ================================

# Load complex time-domain data
#x_postddscic_t = np.loadtxt(data_file, dtype=complex)
with SlabFile(data_file, 'r') as f:
        xi_adc = np.array(f['xi_ADC'])[0]
        xq_adc = np.array(f['xq_ADC'])[0]
        xi_pfb = np.array(f['xi_PFB'])[0]
        xq_pfb = np.array(f['xq_PFB'])[0]
        fs_adc = np.array(f['ADC fs'])[0]
        fs_pfb = np.array(f['PFB fs'])[0]
        fs_ddscic = np.array(f['DDS+CIC fs'])[0]
        qout = np.array(f['Qout'])[0]
        qprod = np.array(f['Qprod'])[0]
        D = np.array(f['Decimation'])[0]  # CIC decimation factor
    
x_postddscic_t = xi_pfb + 1j*xq_pfb
x_postddscic_t = np.array(x_postddscic_t, dtype=complex)

# Effective sampling rate after decimation
fs_d = fs_ch / D
print(f"Effective sampling rate after decimation: {fs_d/1e6:.2f} MHz")

# FFT
X = np.fft.fftshift(np.fft.fft(x_postddscic_t))
f_axis = np.fft.fftshift(np.fft.fftfreq(len(X), 1/fs_d))

# Amplitude in dB
amp_dB = 20*np.log10(np.abs(X)/np.max(np.abs(X)))

# Phase
phase_rad = np.angle(X)

# CIC theoretical amplitude response (normalized)
H_cic = np.sinc(f_axis * D / fs_ch)**N
H_cic_dB = 20*np.log10(H_cic/np.max(H_cic))

# -------------------------------
# Plot amplitude
plt.figure(figsize=(10,5))
plt.plot(f_axis/1e6, amp_dB, label='Measured post-DDSCIC')
plt.plot(f_axis/1e6, H_cic_dB, 'r--', label='CIC theoretical')
plt.xlabel('Frequency [MHz]')
plt.ylabel('Amplitude [dB]')
plt.title(f'Amplitude Spectrum (D={D}, N={N})')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

# -------------------------------
# Plot phase
plt.figure(figsize=(10,5))
plt.plot(f_axis/1e6, phase_rad)
plt.xlabel('Frequency [MHz]')
plt.ylabel('Phase [rad]')
plt.title('Phase Spectrum')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# ================================
# User parameters
# ================================
Pin = '-12'
data_pre = 'D:\\Morgan\\20260206_cooldown\\data\\' + date + '\\00000_' + date + '_ADC+PFB_Cal' + Pin + 'dBm.h5'   # signal before DDS+CIC
data_post = 'D:\\Morgan\\20260206_cooldown\\data\\' + date + '\\00000_' + date + '_DDS+CIC_Cal' + Pin + 'dBm.h5' # signal after DDS+CIC
fs_ch = 256e6       # ADC sampling frequency before decimation
D = 4               # CIC decimation factor
N = 3               # CIC order
# ================================

# Load signals
with SlabFile(data_pre, 'r') as f:
        xi_adc = np.array(f['xi_ADC'])[0]
        xq_adc = np.array(f['xq_ADC'])[0]
        xi_pfb = np.array(f['xi_PFB'])[0]
        xq_pfb = np.array(f['xq_PFB'])[0]
        fs_adc = np.array(f['ADC fs'])[0]
        fs_pfb = np.array(f['PFB fs'])[0]
        qout = np.array(f['Qout'])[0]

x_pre = xi_pfb + 1j*xq_pfb
x_pre = np.array(x_pre, dtype=complex)

with SlabFile(data_post, 'r') as f:
        xi_adc = np.array(f['xi_ADC'])[0]
        xq_adc = np.array(f['xq_ADC'])[0]
        xi_pfb = np.array(f['xi_PFB'])[0]
        xq_pfb = np.array(f['xq_PFB'])[0]
        fs_adc = np.array(f['ADC fs'])[0]
        fs_pfb = np.array(f['PFB fs'])[0]
        fs_ddscic = np.array(f['DDS+CIC fs'])[0]
        qout = np.array(f['Qout'])[0]
        qprod = np.array(f['Qprod'])[0]
        D = np.array(f['Decimation'])[0]
    
x_post = xi_pfb + 1j*xq_pfb
x_post = np.array(x_post, dtype=complex)

# Effective sampling rate after decimation
fs_d = fs_ch / D
print(f"Effective sampling rate after decimation: {fs_d/1e6:.2f} MHz")

# --- FFT for 'Before' (Full Rate) ---
X_pre = np.fft.fftshift(np.fft.fft(x_pre))
# Use the full ADC sampling rate here
f_axis_pre = np.fft.fftshift(np.fft.fftfreq(len(x_pre), 1/fs_ch))

# --- FFT for 'After' (Decimated Rate) ---
X_post = np.fft.fftshift(np.fft.fft(x_post))
# Use the decimated sampling rate here
f_axis_post = np.fft.fftshift(np.fft.fftfreq(len(x_post), 1/fs_d))

# Normalize amplitudes for comparison
amp_pre_dB = 20*np.log10(np.abs(X_pre)/np.max(np.abs(X_pre)))
amp_post_dB = 20*np.log10(np.abs(X_post)/np.max(np.abs(X_post)))

# Phase
phase_pre_rad = np.angle(X_pre)
phase_post_rad = np.angle(X_post)

# CIC theoretical response
H_cic = np.sinc(f_axis * D / fs_ch)**N
H_cic_dB = 20*np.log10(H_cic/np.max(H_cic))

# -------------------------------
# Plot amplitude comparison
plt.figure(figsize=(10,5))
# Plot 'Before' against the 256MHz-based axis
plt.plot(f_axis_pre/1e6, amp_pre_dB, label='Before DDS+CIC (Full BW)', alpha=0.7)
# Plot 'After' against the 32MHz-based axis
plt.plot(f_axis_post/1e6, amp_post_dB, label='After DDS+CIC (Decimated)', linewidth=2)
#plt.plot(f_axis_pre/1e6, amp_pre_dB, label='Before DDS+CIC')
#plt.plot(f_axis_post/1e6, amp_post_dB, label='After DDS+CIC')
plt.plot(f_axis/1e6, H_cic_dB, 'r--', label='3rd-order CIC theoretical')
plt.xlabel('Frequency [MHz]')
plt.ylabel('Amplitude [dB]')
plt.title('Amplitude Spectrum Comparison')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

# -------------------------------
# Plot phase comparison
plt.figure(figsize=(10,5))
plt.plot(f_axis_pre/1e6, phase_pre_rad, label='Before DDS+CIC')
plt.plot(f_axis_post/1e6, phase_post_rad, label='After DDS+CIC')
plt.xlabel('Frequency [MHz]')
plt.ylabel('Phase [rad]')
plt.title('Phase Spectrum Comparison')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ===============================
# LOAD DATA
# ===============================
Pin = '-12'

data_pre = 'D:\\Morgan\\20260206_cooldown\\data\\' + date + '\\00000_' + date + '_ADC+PFB_Cal' + Pin + 'dBm.h5'   # signal before DDS+CIC
data_post = 'D:\\Morgan\\20260206_cooldown\\data\\' + date + '\\00000_' + date + '_DDS+CIC_Cal' + Pin + 'dBm.h5' # signal after DDS+CIC

with SlabFile(data_pre, 'r') as f:
        xi_adc = np.array(f['xi_ADC'])[0]
        xq_adc = np.array(f['xq_ADC'])[0]
        xi_pfb = np.array(f['xi_PFB'])[0]
        xq_pfb = np.array(f['xq_PFB'])[0]
        fs_adc = np.array(f['ADC fs'])[0]
        fs_pfb = np.array(f['PFB fs'])[0]
        qout = np.array(f['Qout'])[0]
    
Fs_pre  = fs_pfb*1e6
pre = xi_pfb + 1j*xq_pfb
pre = np.array(pre, dtype=complex)

with SlabFile(data_post, 'r') as f:
        xi_adc = np.array(f['xi_ADC'])[0]
        xq_adc = np.array(f['xq_ADC'])[0]
        xi_pfb = np.array(f['xi_PFB'])[0]
        xq_pfb = np.array(f['xq_PFB'])[0]
        fs_adc = np.array(f['ADC fs'])[0]
        fs_pfb = np.array(f['PFB fs'])[0]
        fs_ddscic = np.array(f['DDS+CIC fs'])[0]
        qout = np.array(f['Qout'])[0]
        qprod = np.array(f['Qprod'])[0]
        D = np.array(f['Decimation'])[0]
Fs_post  = fs_ddscic*1e6
print(Fs_post)

post = xi_pfb + 1j*xq_pfb
post = np.array(post, dtype=complex)

# if complex IQ stored as I,Q
#if pre.ndim > 1:
#    pre = pre[:,0] + 1j*pre[:,1]

#if post.ndim > 1:
#    post = post[:,0] + 1j*post[:,1]

# ===============================
# TIME DOMAIN
# ===============================

plt.figure(figsize=(10,4))
plt.plot(np.real(pre[:2000000]), label="pre")
plt.plot(np.real(post[:2000000]), label="post")
plt.legend()
plt.title("Time domain comparison")
plt.show()

# ===============================
# FFT PRE
# ===============================

N = len(pre)

fft_pre = np.fft.fftshift(np.fft.fft(pre))
f_pre = np.fft.fftshift(np.fft.fftfreq(N,1/Fs_pre))

plt.figure(figsize=(7,5))
plt.plot(f_pre/1e6,20*np.log10(np.abs(fft_pre)))
plt.title("FFT before DDS+CIC")
plt.xlabel("MHz")
plt.ylabel("dB")
plt.show()

# ===============================
# FFT POST
# ===============================

N = len(post)

fft_post = np.fft.fftshift(np.fft.fft(post))
f_post = np.fft.fftshift(np.fft.fftfreq(N,1/Fs_post))

plt.figure(figsize=(7,5))
plt.plot(f_post/1e6,20*np.log10(np.abs(fft_post)))
plt.title("FFT after DDS+CIC")
plt.xlabel("MHz")
plt.ylabel("dB")
plt.show()

# ===============================
# AMPLITUDE
# ===============================

amp_pre  = np.abs(pre)
amp_post = np.abs(post)

print("Mean amplitude pre :", np.mean(amp_pre))
print("Mean amplitude post:", np.mean(amp_post))

# ===============================
# PHASE
# ===============================

phase_pre  = np.unwrap(np.angle(pre))
phase_post = np.unwrap(np.angle(post))

plt.figure(figsize=(8,4))
#plt.plot(phase_pre[:10000], label="pre")
plt.plot(phase_post[:60000], label="post")
plt.legend()
plt.title("Phase evolution")
plt.show()

gain_factor = np.mean(np.abs(x_post)) / np.mean(np.abs(x_pre))
print(f"Block gain: {gain_factor:.2f}")

# Tomamos la diferencia de fase entre el último y el primer punto
delta_phase = phase_post[-1] - phase_post[0]
delta_time = len(phase_post) / fs_d

# Calculamos el error de frecuencia en Hz
f_error = delta_phase / (2 * np.pi * delta_time)
print(f"Residual frequency error: {f_error:.2f} Hz")

In [ ]:
# ==========================================
# SYSTEM PARAMETERS
# ==========================================
ANALOG_ATTEN = 16.59  # dB analog attenuation

# PFB
PFB_CHANNELS = 8

# Pre-DDS calibration
CAL_CONSTANT_PRE = -3.28
K_ADC = PFB_SCALING + ANALOG_ATTEN - CAL_CONSTANT_PRE
print(K_ADC)

# ==========================================
# LOAD DATA
# ==========================================
date = '20260327'
file_paths = [
    '00000_' + date + '_DDS+CIC_Cal-40dBm.h5',
    '00000_' + date + '_DDS+CIC_Cal-38dBm.h5',
    '00000_' + date + '_DDS+CIC_Cal-35dBm.h5',
    '00000_' + date + '_DDS+CIC_Cal-32dBm.h5',
    '00000_' + date + '_DDS+CIC_Cal-30dBm.h5',
    '00000_' + date + '_DDS+CIC_Cal-28dBm.h5',
    '00000_' + date + '_DDS+CIC_Cal-25dBm.h5',
    '00000_' + date + '_DDS+CIC_Cal-22dBm.h5',
    '00000_' + date + '_DDS+CIC_Cal-20dBm.h5',
    '00000_' + date + '_DDS+CIC_Cal-18dBm.h5',
    '00000_' + date + '_DDS+CIC_Cal-15dBm.h5',
    '00000_' + date + '_DDS+CIC_Cal-12dBm.h5',
    '00000_' + date + '_DDS+CIC_Cal-10dBm.h5',
    '00000_' + date + '_DDS+CIC_Cal-8dBm.h5',
    '00000_' + date + '_DDS+CIC_Cal-5dBm.h5',
    '00000_' + date + '_DDS+CIC_Cal0dBm.h5',
    '00000_' + date + '_DDS+CIC_Cal5dBm.h5',
    '00000_' + date + '_DDS+CIC_Cal6dBm.h5',
    '00000_' + date + '_DDS+CIC_Cal7dBm.h5',
    '00000_' + date + '_DDS+CIC_Cal8dBm.h5',
    '00000_' + date + '_DDS+CIC_Cal10dBm.h5',
]#

FS = 2**15
tone_bins = 3
measurements = []

# ==========================================
# PROCESS FILES
# ==========================================
for file in file_paths:
    Pin_gen = float(re.search(r'(-?\+?\d+)dBm', file).group(1))
    with SlabFile('D:\\Morgan\\20260206_cooldown\\data\\' + date + '\\' + file, 'r') as f:
        xi_adc = np.array(f['xi_ADC'])[0]
        xq_adc = np.array(f['xq_ADC'])[0]
        xi_pfb = np.array(f['xi_PFB'])[0]
        xq_pfb = np.array(f['xq_PFB'])[0]
        fs_adc = np.array(f['ADC fs'])[0]
        fs_pfb = np.array(f['PFB fs'])[0]
        fs_ddscic = np.array(f['DDS+CIC fs'])[0]
        qout = np.array(f['Qout'])[0]
        qprod = np.array(f['Qprod'])[0]
        D = np.array(f['Decimation'])[0]

    PFB_QOUT = qout
    PFB_SCALING = 20*np.log10(2**PFB_QOUT)  # dB scaling due to PFB output

    # CIC
    N = 3   # Order
    R = D
    M = 1
    CIC_GAIN = (R*M)**N
    CIC_GAIN_DB = 20*np.log10(CIC_GAIN)
    fs_out = fs_pfb / D

    # DDS+CIC configuration for this measurement
    QPROD = qprod               # The configured DDS product quantization used in this measurement
    QPROD_DDS_INTERNAL = 24  # Internal DDS product width
    CIC_OUTPUT_BITS = 16      # Actual output bits

    x = xi_pfb + 1j*xq_pfb
    x = np.array(x, dtype=np.complex128)
    
    N_samples = len(x)
    max_val = np.max(np.abs(x))
    std_val = np.std(np.abs(x))
    
    # Window & FFT
    window = np.hanning(N_samples)
    coherent_gain = np.sum(window)/N_samples
    xw = x*window
    fft_complex = np.fft.fftshift(np.fft.fft(xw)/N_samples)
    freqs = np.fft.fftshift(np.fft.fftfreq(N_samples, 1/fs_out))
    mag = np.abs(fft_complex)/coherent_gain
    
    # Peak detection & tone power
    peak_idx = np.argmax(mag)
    f_peak = freqs[peak_idx]
    idx0, idx1 = peak_idx - tone_bins, peak_idx + tone_bins + 1
    tone_power = np.sum(mag[idx0:idx1]**2)
    P_output_dbfs = 10*np.log10(tone_power/(FS**2))
    
    # Phase
    tone_phase = np.angle(fft_complex[peak_idx])
    
    # SNR & SFDR
    noise_mask = np.ones(len(mag), dtype=bool)
    noise_mask[idx0:idx1] = False
    noise_power = np.mean(mag[noise_mask]**2)
    snr_db = 10*np.log10(tone_power/(len(mag)*noise_power))
    mag_no_tone = mag.copy(); mag_no_tone[idx0:idx1] = 0
    sfdr_db = 10*np.log10(tone_power/np.max(mag_no_tone)**2)
    
    measurements.append({
        'Pin_gen': Pin_gen,
        'P_output_dbfs': P_output_dbfs,
        'f_peak': f_peak,
        'phase': tone_phase,
        'max_val': max_val,
        'std_val': std_val,
        'snr_db': snr_db,
        'sfdr_db': sfdr_db,
        'file': file
    })
    
    saturation_pct = (max_val/FS)*100
    sat_flag = " ⚠️ SAT" if saturation_pct > 90 else ""
    print(f"Pin={Pin_gen:+5.0f} dBm → Out={P_output_dbfs:7.2f} dBFS, Phase={np.degrees(tone_phase):+7.1f}°, SNR={snr_db:5.1f} dB, Peak={saturation_pct:4.1f}%{sat_flag}")

# ==========================================
# CALIBRATION
# ==========================================
linear_meas = measurements[:8]
Pin_vals = np.array([m['Pin_gen'] for m in linear_meas])
Pout_vals = np.array([m['P_output_dbfs'] for m in linear_meas])
coeffs = np.polyfit(Pout_vals, Pin_vals, 1)
slope = coeffs[0]
CAL_CONSTANT_POST = coeffs[1]

print(f"\nLinear fit: Pin = {slope:.3f} × P_out + {CAL_CONSTANT_POST:.2f} dB")

# ==========================================
# DDS+CIC QUANTIZATION AND GAIN ANALYSIS
# ==========================================
print(f"\n{'='*80}")
print("DDS+CIC QUANTIZATION AND GAIN ANALYSIS")
print(f"{'='*80}")

TRUNC_BITS = QPROD_DDS_INTERNAL - CIC_OUTPUT_BITS
TRUNC_DB = 6.02*TRUNC_BITS
DDS_CIC_NET_EFFECT = CAL_CONSTANT_PRE - CAL_CONSTANT_POST
UNACCOUNTED_LOSS = CIC_GAIN_DB - DDS_CIC_NET_EFFECT
residual_loss = UNACCOUNTED_LOSS - TRUNC_DB

print(f"Configured QPROD = {QPROD} (used for this data)")
print(f"Internal DDS width: {QPROD_DDS_INTERNAL} bits, CIC output: {CIC_OUTPUT_BITS} bits")
print(f"Truncation effect: {TRUNC_BITS} bits → {TRUNC_DB:.2f} dB")
print(f"Theoretical CIC gain: {CIC_GAIN_DB:.2f} dB")
print(f"Measured net gain: {DDS_CIC_NET_EFFECT:.2f} dB (Theoretical CIC gain - Truncation effect: {CIC_GAIN_DB-TRUNC_DB:.2f} dB)")
print(f"Model mismatch: {residual_loss:.2f} dB (likely internal scaling/truncation effects)")

# ==========================================
# PHASE ANALYSIS
# ==========================================
Pin_list = [m['Pin_gen'] for m in measurements]
phases_deg = [np.degrees(m['phase']) for m in measurements]
phase_mean = np.mean(phases_deg)
phase_std = np.std(phases_deg)
phase_power_corr = np.corrcoef(Pin_list, phases_deg)[0,1]

print(f"\nPhase mean: {phase_mean:.2f}°, std dev: {phase_std:.2f}°")
print(f"Phase-Power correlation: {phase_power_corr:.3f}")

# ==========================================
# PLOTTING
# ==========================================
fig = plt.figure(figsize=(18, 14))
gs = fig.add_gridspec(4, 3, hspace=0.35, wspace=0.35)

# Calibration
ax1 = fig.add_subplot(gs[0, 0])
Pin_recovered = [m['P_output_dbfs'] + CAL_CONSTANT_POST for m in measurements]
errors = np.array(Pin_recovered) - np.array(Pin_list)
ax1.plot(Pin_list, Pin_recovered, 'o-', ms=8, lw=2, color='blue', label='Recovered')
ax1.plot(Pin_list, Pin_list, 'k--', lw=2, label='Ideal')
ax1.set_xlabel('True Input Power (dBm)', fontsize=11)
ax1.set_ylabel('Recovered Power (dBm)', fontsize=11)
ax1.set_title('Post-DDS+CIC Calibration', fontsize=12, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Calibration Error
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(Pin_list, errors, 'o-', ms=8, lw=2, color='green')
ax2.axhline(0, color='k', ls='--', lw=2)
ax2.fill_between(Pin_list, -1, 1, alpha=0.15, color='green')
ax2.set_xlabel('Input Power (dBm)', fontsize=11)
ax2.set_ylabel('Error (dB)', fontsize=11)
ax2.set_title(f'Error (σ={np.std(errors[:8]):.3f} dB)', fontsize=12, fontweight='bold')
ax2.grid(True, alpha=0.3)

# Phase vs Power (KEY DIAGNOSTIC)
ax3 = fig.add_subplot(gs[0, 2])
snr_list = [m['snr_db'] for m in measurements]
scatter = ax3.scatter(Pin_list, phases_deg, c=snr_list, s=100, 
                     cmap='viridis', edgecolors='k', linewidth=1)
ax3.axhline(phase_mean, color='r', ls='--', lw=2, alpha=0.7)
if abs(phase_power_corr) > 0.5:
    coeffs_phase = np.polyfit(Pin_list, phases_deg, 1)
    p_fit = np.poly1d(coeffs_phase)
    ax3.plot(Pin_list, p_fit(Pin_list), 'r-', lw=2, alpha=0.5,
             label=f'r={phase_power_corr:.2f}')
    ax3.legend()
ax3.set_xlabel('Input Power (dBm)', fontsize=11)
ax3.set_ylabel('Phase (degrees)', fontsize=11)
ax3.set_title(f'Phase vs Power (σ={phase_std:.1f}°)', fontsize=12, fontweight='bold')
plt.colorbar(scatter, ax=ax3, label='SNR (dB)')
ax3.grid(True, alpha=0.3)

# Saturation Check
ax4 = fig.add_subplot(gs[1, 0])
max_vals = [m['max_val'] for m in measurements]
ax4.plot(Pin_list, max_vals, 'o-', ms=8, lw=2, color='red')
ax4.axhline(FS, color='k', ls='-', lw=2, label='Full Scale')
ax4.axhline(0.9*FS, color='orange', ls='--', lw=2, label='90% FS')
ax4.set_xlabel('Input Power (dBm)', fontsize=11)
ax4.set_ylabel('Peak Amplitude', fontsize=11)
ax4.set_title('Saturation Check', fontsize=12, fontweight='bold')
ax4.legend()
ax4.grid(True, alpha=0.3)

# SNR
ax5 = fig.add_subplot(gs[1, 1])
ax5.plot(Pin_list, snr_list, 'o-', ms=8, lw=2, color='green')
ax5.set_xlabel('Input Power (dBm)', fontsize=11)
ax5.set_ylabel('SNR (dB)', fontsize=11)
ax5.set_title('Signal-to-Noise Ratio', fontsize=12, fontweight='bold')
ax5.grid(True, alpha=0.3)

# SFDR
ax6 = fig.add_subplot(gs[1, 2])
sfdr_list = [m['sfdr_db'] for m in measurements]
ax6.plot(Pin_list, sfdr_list, 'o-', ms=8, lw=2, color='purple')
ax6.set_xlabel('Input Power (dBm)', fontsize=11)
ax6.set_ylabel('SFDR (dB)', fontsize=11)
ax6.set_title('Spurious-Free Dynamic Range', fontsize=12, fontweight='bold')
ax6.grid(True, alpha=0.3)

# Phase Histogram
ax7 = fig.add_subplot(gs[2, 0])
ax7.hist(phases_deg, bins=15, edgecolor='k', alpha=0.7, color='purple')
ax7.axvline(phase_mean, color='r', ls='--', lw=2, label=f'μ={phase_mean:.1f}°')
ax7.axvline(phase_mean - phase_std, color='orange', ls=':', lw=1.5, alpha=0.7)
ax7.axvline(phase_mean + phase_std, color='orange', ls=':', lw=1.5, alpha=0.7)
ax7.set_xlabel('Phase (degrees)', fontsize=11)
ax7.set_ylabel('Count', fontsize=11)
ax7.set_title(f'Phase Distribution (σ={phase_std:.1f}°)', fontsize=12, fontweight='bold')
ax7.legend()
ax7.grid(True, alpha=0.3, axis='y')

# Frequency Stability
ax8 = fig.add_subplot(gs[2, 1])
freqs_peak = [m['f_peak']/1e6 for m in measurements]
ax8.plot(Pin_list, freqs_peak, 'o-', ms=8, lw=2, color='orange')
ax8.axhline(np.mean(freqs_peak), color='r', ls='--', lw=2, 
            label=f'μ={np.mean(freqs_peak):.4f} MHz')
ax8.set_xlabel('Input Power (dBm)', fontsize=11)
ax8.set_ylabel('Peak Frequency (MHz)', fontsize=11)
ax8.set_title('Frequency After DDS', fontsize=12, fontweight='bold')
ax8.legend()
ax8.grid(True, alpha=0.3)

# IQ Constellation
ax9 = fig.add_subplot(gs[2, 2])
colors = plt.cm.plasma(np.linspace(0, 1, len(measurements)))
for i, (m, c) in enumerate(zip(measurements[::2], colors[::2])):  # Plot every other
    with SlabFile('D:\\Morgan\\20260206_cooldown\\data\\' + date + '\\' + m['file'], 'r') as f:
        xi_adc = np.array(f['xi_ADC'])[0]
        xq_adc = np.array(f['xq_ADC'])[0]
        xi_pfb = np.array(f['xi_PFB'])[0]
        xq_pfb = np.array(f['xq_PFB'])[0]
        fs_adc = np.array(f['ADC fs'])[0]
        fs_pfb = np.array(f['PFB fs'])[0]
        fs_ddscic = np.array(f['DDS+CIC fs'])[0]
        qout = np.array(f['Qout'])[0]
        qprod = np.array(f['Qprod'])[0]
        D = np.array(f['Decimation'])[0]
    x_data = xi_pfb + 1j*xq_pfb
    x_data = np.array(x_data, dtype=np.complex128)
    subsample = max(1, len(x_data) // 400)
    label = f"{m['Pin_gen']:+.0f} dBm" if i % 2 == 0 else ""
    ax9.scatter(np.real(x_data[::subsample]), np.imag(x_data[::subsample]),
               alpha=0.3, s=1, color=c, label=label)
ax9.set_xlabel('I (Real)', fontsize=11)
ax9.set_ylabel('Q (Imaginary)', fontsize=11)
ax9.set_title('IQ Constellation', fontsize=12, fontweight='bold')
ax9.legend(fontsize=8, markerscale=5, ncol=2)
ax9.grid(True, alpha=0.3)
ax9.axis('equal')

# Spectrum (middle power)
ax10 = fig.add_subplot(gs[3, :2])
mid_idx = len(measurements) // 2
with SlabFile('D:\\Morgan\\20260206_cooldown\\data\\' + date + '\\' + measurements[mid_idx]['file'], 'r') as f:
    xi_adc = np.array(f['xi_ADC'])[0]
    xq_adc = np.array(f['xq_ADC'])[0]
    xi_pfb = np.array(f['xi_PFB'])[0]
    xq_pfb = np.array(f['xq_PFB'])[0]
    fs_adc = np.array(f['ADC fs'])[0]
    fs_pfb = np.array(f['PFB fs'])[0]
    fs_ddscic = np.array(f['DDS+CIC fs'])[0]
    qout = np.array(f['Qout'])[0]
    qprod = np.array(f['Qprod'])[0]
    D = np.array(f['Decimation'])[0]
x_mid = xi_pfb + 1j*xq_pfb
x_mid = np.array(x_mid, dtype=np.complex128)
window_mid = np.hanning(len(x_mid))
X_mid = np.fft.fftshift(np.fft.fft(x_mid * window_mid) / len(x_mid))
f_mid = np.fft.fftshift(np.fft.fftfreq(len(x_mid), 1/fs_out))
mag_mid_dbm = 20*np.log10(np.abs(X_mid)/FS) + CAL_CONSTANT_POST
ax10.plot(f_mid/1e6, mag_mid_dbm, lw=1, alpha=0.7)
ax10.axvline(0, color='r', ls='--', lw=2, alpha=0.5, label='DC (after DDS)')
ax10.set_xlabel('Frequency (MHz)', fontsize=11)
ax10.set_ylabel('Power (dBm)', fontsize=11)
ax10.set_title(f'Calibrated Spectrum (Pin={Pin_list[mid_idx]:+.0f} dBm)', 
              fontsize=12, fontweight='bold')
ax10.legend()
ax10.grid(True, alpha=0.3)
ax10.set_xlim([-fs_out/2/1e6, fs_out/2/1e6])

# Summary Table
ax11 = fig.add_subplot(gs[3, 2])
ax11.axis('off')
summary_text = f"""
CALIBRATION SUMMARY

Pre-DDS:  {CAL_CONSTANT_PRE:.2f} dB
Post-DDS: {CAL_CONSTANT_POST:.2f} dB
Net DDS+CIC: {DDS_CIC_NET_EFFECT:.2f} dB

CIC gain: {CIC_GAIN_DB:.2f} dB
Unexplained: {UNACCOUNTED_LOSS:.2f} dB

Accuracy: ±{np.std(errors[:8]):.3f} dB
Phase σ: {phase_std:.2f}°
SNR range: {np.min(snr_list):.1f}-{np.max(snr_list):.1f} dB

Phase-Power corr: {phase_power_corr:.3f}
"""
ax11.text(0.1, 0.5, summary_text, transform=ax11.transAxes,
         fontsize=11, verticalalignment='center', fontfamily='monospace',
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.savefig('D:\\Morgan\\20260206_cooldown\\data\\' + date + '\\images\\ddscic_complete_analysis.pdf', dpi=300, bbox_inches='tight')
plt.show()

# ==========================================
# FINAL REPORT
# ==========================================
print(f"\n{'='*80}")
print("FINAL DIAGNOSTIC REPORT")
print(f"{'='*80}")

print(f" CALIBRATION:")
errors = np.array(Pin_recovered) - np.array(Pin_list)
print(f"  • Linear fit accuracy: ±{np.std(errors[:8]):.3f} dB")
print(f"  • Pin (dBm) ≈ P_out (dBFS) + {CAL_CONSTANT_POST:.2f}")

print(f"\n DDS+CIC CHAIN (QPROD={QPROD}):")
print(f"  • Theoretical CIC gain: {CIC_GAIN_DB:.2f} dB")
print(f"  • Truncation effect: {TRUNC_DB:.2f} dB")
print(f"  • Measured net gain: {DDS_CIC_NET_EFFECT:.2f} dB")
print(f"  • Model mismatch: {residual_loss:.2f} dB")

print(f"\n PHASE:")
if phase_std < 10:
    print(f"   Excellent: {phase_std:.2f}° std dev")
elif phase_std < 20:
    print(f"   Good: {phase_std:.2f}° std dev")
elif abs(phase_power_corr) > 0.7:
    print(f"  ! Phase depends on power (r={phase_power_corr:.2f})")
else:
    print(f"  ! High variation ({phase_std:.2f}°) without power correlation")

## ==========================================
## FULL DSP CHAIN GAIN ANALYSIS
## ==========================================
#print(f"\n{'='*80}")
#print("FULL DSP CHAIN GAIN ANALYSIS")
#print(f"{'='*80}")
#
## PFB scaling
#PFB_GAIN = 2**(-PFB_QOUT)
#PFB_GAIN_DB = 20*np.log10(PFB_GAIN)
#
## CIC gain
#CIC_GAIN_DB = 20*np.log10((R*M)**N)
#
## truncation
#TRUNC_BITS = QPROD_DDS_INTERNAL - CIC_OUTPUT_BITS
#TRUNC_GAIN_DB = -6.02 * TRUNC_BITS
#
## theoretical total
#TOTAL_THEORY_DB = PFB_GAIN_DB + CIC_GAIN_DB + TRUNC_GAIN_DB
#
## measured
#MEASURED_NET_GAIN = CAL_CONSTANT_PRE - CAL_CONSTANT_POST
#
#print("PFB:")
#print(f"  scaling = 2^-{PFB_QOUT}")
#print(f"  gain = {PFB_GAIN_DB:.2f} dB")
#
#print("\nCIC:")
#print(f"  R={R}, M={M}, N={N}")
#print(f"  gain = {CIC_GAIN_DB:.2f} dB")
#
#print("\nDDS+CIC truncation:")
#print(f"  internal width = {QPROD_DDS_INTERNAL} bits")
#print(f"  output width   = {CIC_OUTPUT_BITS} bits")
#print(f"  truncation     = {TRUNC_BITS} bits")
#print(f"  trunc gain     = {TRUNC_GAIN_DB:.2f} dB")
#
#print("\nTOTAL THEORETICAL DSP GAIN:")
#print(f"  G_total = {TOTAL_THEORY_DB:.2f} dB")
#
#print("\nMEASURED GAIN:")
#print(f"  G_measured = {MEASURED_NET_GAIN:.2f} dB")
#
#gain_error = MEASURED_NET_GAIN - TOTAL_THEORY_DB
#
#print("\nGAIN ERROR:")
#print(f"  error = {gain_error:.2f} dB")

# WXFFT

In [ ]:
# ==========================================
# SYSTEM PARAMETERS
# ==========================================
N = 65536

# FFT input is the 16-bit DSP output (not the 12-bit ADC)
N_bits = 16
FS = 2**(N_bits - 1)

print("="*80)
print("SYSTEM PARAMETERS")
print("="*80)
print(f"FFT size: {N}")
print(f"DSP resolution: {N_bits} bits")
print(f"Full-scale: {FS}")
print(f"Frequency resolution: {Fs/N:.3f} Hz")
#print(f"FFT calibration constant: {C_fft:.2f} dB")
print("="*80)

# ==========================================
# FIND AND SORT FILES
# ==========================================
date = '20260327'
file_paths = [
    '00000_' + date + '_WXFFT_Cal-40dBm.h5',
    '00000_' + date + '_WXFFT_Cal-38dBm.h5',
    '00000_' + date + '_WXFFT_Cal-35dBm.h5',
    '00000_' + date + '_WXFFT_Cal-32dBm.h5',
    '00000_' + date + '_WXFFT_Cal-30dBm.h5',
    '00000_' + date + '_WXFFT_Cal-28dBm.h5',
    '00000_' + date + '_WXFFT_Cal-25dBm.h5',
    '00000_' + date + '_WXFFT_Cal-22dBm.h5',
    '00000_' + date + '_WXFFT_Cal-20dBm.h5',
    '00000_' + date + '_WXFFT_Cal-18dBm.h5',
    '00000_' + date + '_WXFFT_Cal-15dBm.h5',
    '00000_' + date + '_WXFFT_Cal-12dBm.h5',
    '00000_' + date + '_WXFFT_Cal-10dBm.h5',
    '00000_' + date + '_WXFFT_Cal-8dBm.h5',
    '00000_' + date + '_WXFFT_Cal-5dBm.h5',
    '00000_' + date + '_WXFFT_Cal0dBm.h5',
    '00000_' + date + '_WXFFT_Cal5dBm.h5',
    '00000_' + date + '_WXFFT_Cal6dBm.h5',
    '00000_' + date + '_WXFFT_Cal7dBm.h5',
    '00000_' + date + '_WXFFT_Cal8dBm.h5',
    '00000_' + date + '_WXFFT_Cal10dBm.h5',
]#

labels_list  = ['-40', '-38', '-35', '-32', '-30', '-28', '-25', '-22', '-20', '-18', '-15', '-12', '-10', '-8', '-5', '0', '5', '6', '7', '8', '10']

for i,(file,label) in enumerate(zip(file_paths,labels_list)):
    print(f"  {i+1}. {label}: {file}")

# ==========================================
# PREPARE PLOTTING
# ==========================================
colors = plt.cm.viridis(np.linspace(0,1,len(file_paths)))
fig, axes = plt.subplots(2,1,figsize=(14,10))

freq = np.fft.fftshift(np.fft.fftfreq(N,1/Fs))

# ==========================================
# PROCESS EACH FILE
# ==========================================
peak_info = []

for i,(file_path,label) in enumerate(zip(file_paths,labels_list)):

    print(f"\nProcessing: {label}")


    with SlabFile('D:\\Morgan\\20260206_cooldown\\data\\' + date + '\\' + file_path, 'r') as f:
        xi_adc = np.array(f['xi_ADC'])[0]
        xq_adc = np.array(f['xq_ADC'])[0]
        xi_pfb = np.array(f['xi_PFB'])[0]
        xq_pfb = np.array(f['xq_PFB'])[0]
        P_k = np.array(f['x_WXFFT'])[0]
        fs_adc = np.array(f['ADC fs'])[0]
        fs_pfb = np.array(f['PFB fs'])[0]
        Fs = np.array(f['DDS+CIC fs'])[0]
        qout = np.array(f['Qout'])[0]
        qprod = np.array(f['Qprod'])[0]
        D = np.array(f['Decimation'])[0]  # CIC decimation factor
    if i == 0:
        print(f"Sampling frequency: {Fs/1e6:.2f} MHz")
    # Align spectrum with frequency axis
    #P_k = np.fft.fftshift(P_k)
    print(f'Qout: {qout}, Qprod: {qprod}')

    # FFT calibration constant (measured)
    C_fft = 32.94 +  20*np.log10(((D)**3)*(2**(-qprod)))

    # Recover magnitude
    X_mag = np.sqrt(P_k)

    # Recover tone amplitude
    A = X_mag / N

    # Convert to dBFS
    A_dBFS = 20*np.log10(A/FS + 1e-20)

    # PSD
    enbw_hz = 1.5*(Fs/N)
    PSD_dBFS_Hz = A_dBFS - 10*np.log10(enbw_hz)

    # Peak detection
    peak_idx = np.argmax(A_dBFS)

    peak_freq = freq[peak_idx]
    peak_dBFS = A_dBFS[peak_idx]

    # Estimate RF input power using FFT calibration
    pin_est = peak_dBFS + C_fft

    peak_info.append({
        'label': label,
        'pin_est': pin_est,
        'peak_freq': peak_freq,
        'peak_dBFS': peak_dBFS,
        'peak_linear': A[peak_idx]
    })

    # Plot amplitude
    axes[0].plot(freq/1e6,A_dBFS,
                 color=colors[i],
                 label=f"{label} (Pin≈{pin_est:.1f} dBm)", zorder = len(labels_list) - i)

    # Plot PSD
    axes[1].plot(freq/1e6,PSD_dBFS_Hz,color=colors[i],label=label, zorder = len(labels_list) - i)

# ==========================================
# FINALIZE PLOTS
# ==========================================
axes[0].set_title(f"Amplitude Spectrum (FFT Accumulator) | Fs = {Fs/1e6:.2f} MHz",
                  fontsize=13,fontweight='bold')
axes[0].set_ylabel("Amplitude [dBFS]",fontsize=12)
axes[0].grid(True,alpha=0.4)
axes[0].set_ylim([None,5])
axes[0].legend(fontsize=9,loc='upper right',ncol=2)
axes[0].set_xlim([freq[0]/1e6,freq[-1]/1e6])

axes[1].set_xlabel("Frequency [MHz]",fontsize=12)
axes[1].set_ylabel("PSD [dBFS/Hz]",fontsize=12)
axes[1].set_title("Power Spectral Density",fontsize=13,fontweight='bold')
axes[1].grid(True,alpha=0.4)
axes[1].legend(fontsize=9,loc='upper right',ncol=2)
axes[1].set_xlim([freq[0]/1e6,freq[-1]/1e6])

plt.tight_layout()
plt.savefig('D:\\Morgan\\20260206_cooldown\\data\\' + date + '\\images\\FFT_ACC_AmpdBFS_and_PSD.png',dpi=300,bbox_inches='tight')
plt.show()

# ==========================================
# PRINT SUMMARY TABLE
# ==========================================
print("\n"+"="*80)
print("PEAK ANALYSIS SUMMARY")
print("="*80)
print(f"{'Input Power (est)':<18} {'Peak Freq (MHz)':<18} {'Peak (dBFS)':<15} {'Peak Linear':<15}")
print("-"*80)

for info in peak_info:
    print(f"{info['pin_est']:>8.2f} dBm {info['peak_freq']/1e6:>16.3f} {info['peak_dBFS']:>14.2f} {info['peak_linear']:>14.6f}")

print("="*80)

# ==========================================
# ADDITIONAL ANALYSIS
# ==========================================
input_est=[]
peak_dBFS_vals=[]

for info in peak_info:
    input_est.append(info['pin_est'])
    peak_dBFS_vals.append(info['peak_dBFS'])

fig2,(ax1,ax2)=plt.subplots(1,2,figsize=(14,5))

ax1.plot(input_est,peak_dBFS_vals,'o-',markersize=8,linewidth=2,color='blue')
ax1.set_xlabel('Estimated Input Power (dBm)')
ax1.set_ylabel('Peak Amplitude (dBFS)')
ax1.set_title('Peak Amplitude vs Input Power',fontsize=13,fontweight='bold')
ax1.grid(True,alpha=0.3)

if len(input_est)>1:
    input_diffs=np.diff(input_est)
    peak_diffs=np.diff(peak_dBFS_vals)
    gains=peak_diffs/input_diffs

    ax2.plot(input_est[1:],gains,'o-',markersize=8,linewidth=2,color='green')
    ax2.axhline(y=1.0,color='red',linestyle='--',linewidth=2,label='Unity gain')
    ax2.set_xlabel('Input Power (dBm)')
    ax2.set_ylabel('Gain (dB/dB)')
    ax2.set_title('System Gain vs Input Power',fontsize=13,fontweight='bold')
    ax2.legend()
    ax2.grid(True,alpha=0.3)

plt.tight_layout()
plt.savefig('D:\\Morgan\\20260206_cooldown\\data\\' + date + '\\images\\FFT_ACC_Peak_Analysis.png',dpi=300,bbox_inches='tight')
plt.show()

# ==========================================
# ZOOM PLOT
# ==========================================
mid_idx=len(file_paths)//2
with SlabFile('D:\\Morgan\\20260206_cooldown\\data\\' + date + '\\' + file_paths[mid_idx], 'r') as f:
        xi_adc = np.array(f['xi_ADC'])[0]
        xq_adc = np.array(f['xq_ADC'])[0]
        xi_pfb = np.array(f['xi_PFB'])[0]
        xq_pfb = np.array(f['xq_PFB'])[0]
        P_zoom = np.array(f['x_WXFFT'])[0]
        fs_adc = np.array(f['ADC fs'])[0]
        fs_pfb = np.array(f['PFB fs'])[0]
        Fs = np.array(f['DDS+CIC fs'])[0]
        qout = np.array(f['Qout'])[0]
        qprod = np.array(f['Qprod'])[0]
        D = np.array(f['Decimation'])[0]  # CIC decimation factor

X_mag_zoom=np.sqrt(P_zoom)
A_zoom=X_mag_zoom/N
A_dBFS_zoom=20*np.log10(A_zoom/FS+1e-20)

zoom_mask=(np.abs(freq)<=1e6)

plt.figure(figsize=(12,6))
plt.plot(freq[zoom_mask]/1e3,A_dBFS_zoom[zoom_mask],linewidth=2,color='blue')
plt.axvline(x=0,color='red',linestyle='--',linewidth=2,alpha=0.5,label='DC')
plt.xlabel('Frequency [kHz]',fontsize=12)
plt.ylabel('Amplitude [dBFS]',fontsize=12)
plt.title(f'Zoomed Spectrum (±1 MHz) - {labels_list[mid_idx]}',
          fontsize=13,fontweight='bold')
plt.grid(True,alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig('D:\\Morgan\\20260206_cooldown\\data\\' + date + '\\images\\FFT_ACC_Zoom.png',dpi=300,bbox_inches='tight')
plt.show()

print("\nAnalysis complete. Plots saved to images/ directory")

In [ ]:
# ==========================================================
# COMPLETE DSP CHAIN PARAMETERS
# ==========================================================
print("="*80)
print("COMPLETE DSP CHAIN PARAMETERS")
print("="*80)

# Sampling rates
fs_adc = 2048e6
fs_pfb = 256e6
D_cic = 4
fs_cic = fs_pfb / D_cic

# FFT parameters
N_fft = 65536
fs_acc = fs_cic

# FFT numeric full scale (16-bit DSP output)
N_bits_fft = 16
FS = 2**(N_bits_fft - 1)

print(f"ADC rate: {fs_adc/1e6:.0f} MHz")
print(f"PFB output rate: {fs_pfb/1e6:.0f} MHz")
print(f"CIC output rate: {fs_cic/1e6:.0f} MHz")
print(f"FFT size: {N_fft}")
print(f"FFT numeric full scale: {FS}")

# ==========================================================
# DSP SCALING PARAMETERS
# ==========================================================

# Current configuration
QOUT = 6
QPROD = 16

# Reference configuration used during calibration
QOUT_REF = 6
QPROD_REF = 16

# Measured FFT calibration constant for reference config
C_REF = 32.94 -  20*np.log10(1/1024)

# Gain correction
delta_pfb = -6.02*(QOUT - QOUT_REF)
delta_dds = -6.02*(QPROD - QPROD_REF)

C_fft = C_REF + delta_pfb + delta_dds

print("\nCalibration constant:")
print(f"C_fft = {C_fft:.2f} dB")

# ==========================================================
# LOAD DATA
# ==========================================================
date = '20260327'
file_paths = [
    '00000_' + date + '_WXFFT_Cal-40dBm.h5',
    '00000_' + date + '_WXFFT_Cal-38dBm.h5',
    '00000_' + date + '_WXFFT_Cal-35dBm.h5',
    '00000_' + date + '_WXFFT_Cal-32dBm.h5',
    '00000_' + date + '_WXFFT_Cal-30dBm.h5',
    '00000_' + date + '_WXFFT_Cal-28dBm.h5',
    '00000_' + date + '_WXFFT_Cal-25dBm.h5',
    '00000_' + date + '_WXFFT_Cal-22dBm.h5',
    '00000_' + date + '_WXFFT_Cal-20dBm.h5',
    '00000_' + date + '_WXFFT_Cal-18dBm.h5',
    '00000_' + date + '_WXFFT_Cal-15dBm.h5',
    '00000_' + date + '_WXFFT_Cal-12dBm.h5',
    '00000_' + date + '_WXFFT_Cal-10dBm.h5',
    '00000_' + date + '_WXFFT_Cal-8dBm.h5',
    '00000_' + date + '_WXFFT_Cal-5dBm.h5',
    '00000_' + date + '_WXFFT_Cal0dBm.h5',
    '00000_' + date + '_WXFFT_Cal5dBm.h5',
    '00000_' + date + '_WXFFT_Cal6dBm.h5',
    '00000_' + date + '_WXFFT_Cal7dBm.h5',
    '00000_' + date + '_WXFFT_Cal8dBm.h5',
    '00000_' + date + '_WXFFT_Cal10dBm.h5',
]#

labels_list  = ['-40', '-38', '-35', '-32', '-30', '-28', '-25', '-22', '-20', '-18', '-15', '-12', '-10', '-8', '-5', '0', '5', '6', '7', '8', '10']

# ==========================================================
# FREQUENCY AXIS
# ==========================================================

freq = np.fft.fftshift(np.fft.fftfreq(N_fft, 1/fs_acc))

# ==========================================================
# PROCESS DATA
# ==========================================================

colors = plt.cm.viridis(np.linspace(0,1,len(file_paths)))

fig,axes = plt.subplots(3,1,figsize=(14,12))

peak_info=[]

for i,(file,label) in enumerate(zip(file_paths,labels)):

    print("\nProcessing",label)

    with SlabFile('D:\\Morgan\\20260206_cooldown\\data\\' + date + '\\' + file, 'r') as f:
        xi_adc = np.array(f['xi_ADC'])[0]
        xq_adc = np.array(f['xq_ADC'])[0]
        xi_pfb = np.array(f['xi_PFB'])[0]
        xq_pfb = np.array(f['xq_PFB'])[0]
        P = np.array(f['x_WXFFT'])[0]
        fs_adc = np.array(f['ADC fs'])[0]
        fs_pfb = np.array(f['PFB fs'])[0]
        Fs = np.array(f['DDS+CIC fs'])[0]
        qout = np.array(f['Qout'])[0]
        qprod = np.array(f['Qprod'])[0]
        D = np.array(f['Decimation'])[0]  # CIC decimation factor

    #P = np.fft.fftshift(P)

    Xmag = np.sqrt(P)

    A = Xmag / N_fft

    A_dBFS = 20*np.log10(A/FS + 1e-20)

    A_dBm = A_dBFS + C_fft

    # PSD
    enbw_bins = 1.5
    enbw = enbw_bins*(fs_acc/N_fft)

    PSD_dBm_Hz = A_dBm - 10*np.log10(enbw)

    peak_idx = np.argmax(A_dBFS)

    peak_freq = freq[peak_idx]
    peak_dBFS = A_dBFS[peak_idx]
    peak_dBm = A_dBm[peak_idx]

    Pin = float(label.split()[0])

    err = peak_dBm - Pin

    peak_info.append((Pin,peak_dBm,err,peak_dBFS))

    print(f"Peak {peak_dBFS:.2f} dBFS = {peak_dBm:.2f} dBm")
    print(f"Error {err:+.2f} dB")

    axes[0].plot(freq/1e6,A_dBm,color=colors[i],label=label)
    axes[1].plot(freq/1e6,A_dBFS,color=colors[i])
    axes[2].plot(freq/1e6,PSD_dBm_Hz,color=colors[i])

# ==========================================================
# PLOTS
# ==========================================================

axes[0].set_ylabel("Amplitude [dBm]")
axes[0].set_title("Calibrated Spectrum")
axes[0].grid()
axes[0].legend()

axes[1].set_ylabel("Amplitude [dBFS]")
axes[1].set_title("Spectrum (dBFS)")
axes[1].grid()

axes[2].set_ylabel("PSD [dBm/Hz]")
axes[2].set_xlabel("Frequency [MHz]")
axes[2].set_title("Power Spectral Density")
axes[2].grid()

plt.tight_layout()
plt.show()

# ==========================================================
# CALIBRATION VERIFICATION
# ==========================================================

print("\n"+"="*80)
print("CALIBRATION VERIFICATION")
print("="*80)

Pin=[]
Pout=[]
err=[]

for p in peak_info:

    Pin.append(p[0])
    Pout.append(p[1])
    err.append(p[2])

    print(f"{p[0]:>6.1f} dBm   {p[1]:>7.2f} dBm   err {p[2]:>6.2f} dB")

print("\nStatistics")

print("Mean error:",np.mean(err))
print("Std dev:",np.std(err))
print("Max error:",np.max(np.abs(err)))

# ==========================================================
# POWER RECOVERY PLOT
# ==========================================================

plt.figure(figsize=(8,6))

plt.plot(Pin,Pout,'o',label="Recovered")
plt.plot(Pin,Pin,'k--',label="Ideal")

plt.xlabel("Input power (dBm)")
plt.ylabel("Recovered power (dBm)")

plt.title("RF Power Recovery")

plt.legend()
plt.grid()

plt.show()

# ==========================================================
# FINAL SUMMARY
# ==========================================================

print("\n"+"="*80)
print("FINAL SUMMARY")
print("="*80)

print(f"""
DSP Chain

ADC → PFB → DDS → CIC → FFT

Calibration constant

C_fft = {C_fft:.2f} dB

RF power estimation

P_RF(dBm) = P_FFT(dBFS) + C_fft

Accuracy

σ ≈ {np.std(err):.3f} dB
""")

In [ ]:
Int_fname = fname =  'D:\\Morgan\\20260206_cooldown\\data\\20260320\\Integration_Data\\00000_20260320_IntegrationStep_103.h5'
with SlabFile(Int_fname, 'r') as p:
                #print(p.keys())
                f1 = np.array(p['fpts'])[0]
                N = np.array(p['Navg'])[0]
                D = np.array(p['Dec'])[0]
                DDS = np.array(p['DDS_freq'])[0]
                FFT = np.array(p['FFT Resolution'])[0]
                FFT_N = np.array(p['FFT_Length'])[0]
                x = np.array(p['x'])[0]

x = x*(0.5/2**15)**2

# Scale by FFT length.
x = x/(65536)**2

# Spectrum.
F = np.linspace(-FFT/2,FFT/2,len(x))

# Normalize to dBm.
R = 50
Y = 10*np.log10((np.sqrt(x)**2)/(2*R*1e-3))

plt.figure(dpi=150)
plt.plot(f1,Y, '.')
#plt.plot(f1,Y1,'r*')
plt.xlabel("F [MHz]")
plt.ylabel("PSD")
#plt.xlim(-5, 8)
#plt.ylim(-100, -70)
plt.show()

# 20260318- 20260320 Fridge Signal Comp

## Unknown Qout

In [ ]:
Int_fname ='D:\\Morgan\\20260206_cooldown\\data\\20260320\\Integration_Data\\00000_20260320_IntegrationStep_103.h5'
with SlabFile(Int_fname, 'r') as p:
                #print(p.keys())
                f1 = np.array(p['fpts'])[0]
                N = np.array(p['Navg'])[0]
                D = np.array(p['Dec'])[0]
                DDS = np.array(p['DDS_freq'])[0]
                FFT = np.array(p['FFT Resolution'])[0]
                FFT_N = np.array(p['FFT_Length'])[0]
                x = np.array(p['x'])[0]

print(f'Qprod: {16}, Decimation: {D}')
# Scale by FFT length.
x = x/(65536)**2

# Spectrum.
F = np.linspace(-FFT/2,FFT/2,len(x))

plt.figure(dpi=150)
plt.plot(f1,x, '.')
#plt.plot(f1,Y1,'r*')
plt.xlabel("F [MHz]")
plt.ylabel("ADC^2")
#plt.xlim(-5, 8)
#plt.ylim(-100, -70)
plt.title(f'Qprod: {16}, Decimation: {D}')
plt.show()

## Known Qout

In [ ]:
Int_fname ='D:\\Morgan\\20260206_cooldown\\data\\20260401\\00001_20260401_Check_Qout_10000.h5'
with SlabFile(Int_fname, 'r') as f:
        xi_adc = np.array(f['xi_ADC'])[0]
        xq_adc = np.array(f['xq_ADC'])[0]
        xi_pfb = np.array(f['xi_PFB'])[0]
        xq_pfb = np.array(f['xq_PFB'])[0]
        x = np.array(f['x_WXFFT'])[0]
        qout = np.array(f['Qout'])[0]
        qprod = np.array(f['Qprod'])[0]
        D = np.array(f['Dec'])[0]  # CIC decimation factor
    
print(f'Qout: {qout}, Qprod: {qprod}, Decimation: {D}')

# Scale by FFT length.
x = x/(65536)**2

# Spectrum.
F = np.linspace(-FFT/2,FFT/2,len(x))

plt.figure(dpi=150)
plt.plot(f1,x, '.')
#plt.plot(f1,Y1,'r*')
plt.xlabel("F [MHz]")
plt.ylabel("ADC^2")
#plt.xlim(-5, 8)
#plt.ylim(-100, -70)
plt.title(f'Qout: {qout}, Qprod: {qprod}, Decimation: {D}')
plt.show()